# Predicting Employee Attrition and Identifying Flight Risk Factors

**Group Members:** Shania Siew (816039282), Terrence Murray (816038951), Tyler Baksh (816039328), Syam Manchikanti (816041877)

## Introduction

Employee attrition is expensive. When someone leaves unexpectedly, the company deals with disrupted operations, lost productivity, and recruitment costs that can run anywhere from 50% to 200% of that person's annual salary. Despite this, most organizations still handle turnover reactively, scrambling to fill roles after people have already walked out the door.

This project builds a workforce intelligence platform that flips that approach. We use machine learning classifiers and survival analysis to predict which employees are likely to leave and roughly when, giving organizations a 3 to 6 month window to step in. We also dig into the factors behind flight risk (overtime, tenure, job satisfaction, pay) and check whether our models treat employees fairly across protected attributes like gender, age, marital status, and education field.

We work primarily with the **IBM HR Analytics Employee Attrition & Performance** dataset, which contains detailed employee records covering tenure, salary, role, performance ratings, and attrition status. It's well suited for both classification and survival analysis. We supplement it with the **HR Analytics Job Change of Data Scientists** dataset for additional context on workforce mobility and career transitions.

Our deliverables include a predictive attrition model with flight risk tiers, feature importance analysis (Random Forest, LASSO, SHAP) to surface the main attrition drivers, fairness audits across demographic subgroups, an ROI calculator for estimating cost savings from targeted retention efforts, and an interactive dashboard showing survival curves, risk profiles, and department-level attrition trends.

### Domain Context: Flight Risk Indicators and Data Considerations

Workforce analytics research points to several behavioral and organizational signals tied to flight risk: declining productivity, lower motivation, more absences, reluctance to take on new projects, and pulling back from professional development (AIHR, 2024). Mondore (cited by SHRM) suggests that predictive attrition models should draw on demographic data (age, gender, marital status, education, tenure), performance reviews, engagement surveys, workload metrics, PTO and absenteeism records, and compensation data. He emphasizes including compensation so leadership can put a dollar figure on what's at stake if turnover isn't addressed.

**What the IBM dataset covers:**

| Recommended Data Category | Available Features |
|---|---|
| Demographics & tenure | `Age`, `Gender`, `MaritalStatus`, `Education`, `YearsAtCompany`, `DistanceFromHome` |
| Performance metrics | `PerformanceRating` |
| Engagement & satisfaction | `JobSatisfaction`, `EnvironmentSatisfaction`, `RelationshipSatisfaction`, `WorkLifeBalance`, `JobInvolvement` |
| Workload proxies | `OverTime` |
| Compensation & career growth | `MonthlyIncome`, `PercentSalaryHike`, `StockOptionLevel`, `YearsSinceLastPromotion`, `JobLevel` |
| Career mobility | `NumCompaniesWorked`, `TotalWorkingYears`, `TrainingTimesLastYear`, `YearsInCurrentRole` |

**Limitations:** time-series engagement survey data, PTO/absenteeism records, exit survey responses, and granular productivity metrics. These gaps are a known limitation of working with this dataset.

*Sources: [AIHR — Flight Risk Employee](https://www.aihr.com/blog/flight-risk-employee/), [SHRM — How to Identify Company's Flight Risks](https://www.shrm.org/topics-tools/news/technology/how-to-identify-companys-flight-risks)*

## Part 1: Data Exploration and Cleaning

Before we build any models, we need to get familiar with our data. This section covers:

**Data Quality Checks**
- Load and inspect the dataset (schema, types, dimensions)
- Check for missing values and duplicates
- Drop constant and ID columns (`EmployeeCount`, `StandardHours`, `Over18`, `EmployeeNumber`)

**Exploratory Analysis**
- Summary statistics and distribution plots for numeric, ordinal, and categorical features
- Skewness analysis and log transforms for right-tailed features
- Outlier detection using the IQR method

**Preprocessing**
- Inspect the `Attrition` target variable for class imbalance
- Encode categorical variables (binary mapping + one-hot encoding)

### 1.1.1 Loading and Inspecting the Dataset

First, we download and load the IBM HR Analytics Employee Attrition & Performance dataset. Once it's loaded, we take a quick look at the shape, column names, and data types to get a sense of what we're working with.

In [ ]:
import polars as pl
import requests
import os

# Data Directory
DATA_DIR = "data/raw"
FILE_NAME = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
os.makedirs(DATA_DIR, exist_ok=True)

# Download the dataset from source
url: str = "https://www.kaggle.com/api/v1/datasets/download/pavansubhasht/ibm-hr-analytics-attrition-dataset"

# Check if the dataset already exists
dataset_path = os.path.join(DATA_DIR, "ibm_hr_analytics_attrition_dataset.zip")
if os.path.exists(dataset_path):
    print("Dataset already exists. Skipping download.")
else:
    # Stream download the dataset
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(os.path.join(DATA_DIR, "ibm_hr_analytics_attrition_dataset.zip"), "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    else:    print(f"Failed to download dataset. Status code: {response.status_code}")

# Unzip the dataset
import zipfile
with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

In [ ]:
# Load the dataset using Polars
df = pl.read_csv(os.path.join(DATA_DIR, FILE_NAME))

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns\n")

# List all columns with their data types
for col in df.columns:
    print(f"  {col:30s} {str(df[col].dtype)}")

### Ordinal Feature Mappings

Several numeric columns in the dataset represent ordinal categorical variables. The integer encodings correspond to the following labels:

| Column | 1 | 2 | 3 | 4 | 5 |
|--------|---|---|---|---|---|
| **Education** | Below College | College | Bachelor | Master | Doctor |
| **EnvironmentSatisfaction** | Low | Medium | High | Very High | — |
| **JobInvolvement** | Low | Medium | High | Very High | — |
| **JobSatisfaction** | Low | Medium | High | Very High | — |
| **PerformanceRating** | Low | Good | Excellent | Outstanding | — |
| **RelationshipSatisfaction** | Low | Medium | High | Very High | — |
| **WorkLifeBalance** | Bad | Good | Better | Best | — |

These fields are already numerically encoded in a meaningful ordinal order, so they can be used directly in modeling without further transformation.

### 1.1.3 Missing Value Analysis

We check every column for nulls. If any are found, we'll decide whether to impute or drop them.

In [ ]:
# Count nulls per column
null_counts = df.null_count()
total_nulls = sum(null_counts.row(0))

if total_nulls == 0:
    print(f"No missing values found across all {len(df.columns)} columns.")
else:
    # Show only columns with nulls
    for col in df.columns:
        n = df[col].null_count()
        if n > 0:
            print(f"  {col}: {n} nulls ({n / len(df) * 100:.1f}%)")

### 1.1.4 Duplicate Detection

We check for duplicate rows and remove them if any exist.

In [ ]:
# Check for and remove duplicate rows
n_before = len(df)
df = df.unique().sort("EmployeeNumber")  # sort to ensure consistent row order across runs
n_dupes = n_before - len(df)

if n_dupes == 0:
    print(f"No duplicate rows found ({n_before} rows).")
else:
    print(f"Removed {n_dupes} duplicate rows. {n_before} -> {len(df)} rows.")

### 1.1.2 Removing Irrelevant Features

Looking at the columns, `EmployeeNumber` is just a unique row ID, while `EmployeeCount`, `StandardHours`, and `Over18` are the same value for every row. None of these tell us anything useful about attrition, so we drop them.

In [ ]:
drop_cols: list[str] = ["EmployeeCount", "EmployeeNumber", "StandardHours", "Over18"]

df = df.drop(drop_cols)

print(f"Dropped {len(drop_cols)} columns: {drop_cols}")
print(f"Remaining: {df.shape[0]} rows, {df.shape[1]} columns")

### 1.2.1 Summary Statistics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# -- Feature groups --
# Ordinal: integer-coded but represent ranked categories (see mappings above)
ordinal_cols: list[str] = [
    "Education", "EnvironmentSatisfaction", "JobInvolvement", "JobLevel",
    "JobSatisfaction", "PerformanceRating", "RelationshipSatisfaction",
    "StockOptionLevel", "TrainingTimesLastYear", "WorkLifeBalance",
]

# Categorical: string-typed columns
categorical_cols: list[str] = [col for col in df.columns if df[col].dtype == pl.String]

# Numeric: continuous int/float columns, excluding ordinals
numeric_cols: list[str] = [
    col for col in df.columns
    if df[col].dtype in [pl.Int64, pl.Float64] and col not in ordinal_cols
]


# -- Reusable plotting helpers --
def plot_grid(cols, ncols, plot_fn, title, figsize_per_row=4):
    """Create a grid of subplots, calling plot_fn(ax, col) for each column."""
    nrows = -(-len(cols) // ncols)  # ceiling division
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * figsize_per_row))
    axes = np.array(axes).flatten()

    for i, col in enumerate(cols):
        plot_fn(axes[i], col)

    for j in range(len(cols), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


print(f"Numeric features ({len(numeric_cols)}):     {numeric_cols}")
print(f"Ordinal features ({len(ordinal_cols)}):     {ordinal_cols}")
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

In [ ]:
# Histogram + median line for each numeric feature
def plot_numeric_hist(ax, col):
    data = df.get_column(col).to_list()
    median = df.get_column(col).median()
    ax.hist(data, bins=30, edgecolor="white", alpha=0.8)
    ax.axvline(median, color="red", linestyle="--", linewidth=1, label=f"median={median:.0f}")
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)

plot_grid(numeric_cols, 4, plot_numeric_hist, "Numeric Feature Distributions")

In [ ]:
# Bar chart for each ordinal feature (sorted by category value)
def plot_ordinal_bar(ax, col):
    counts = df[col].value_counts().sort(col)
    ax.bar([str(v) for v in counts.get_column(col).to_list()], counts.get_column("count").to_list())
    ax.set_title(col, fontsize=10)

plot_grid(ordinal_cols, 5, plot_ordinal_bar, "Ordinal Feature Distributions", figsize_per_row=3.5)

In [ ]:
# Horizontal bar chart for each categorical feature (sorted by frequency)
def plot_categorical_bar(ax, col):
    counts = df[col].value_counts().sort("count", descending=True)
    ax.barh(counts.get_column(col).to_list(), counts.get_column("count").to_list())
    ax.set_title(col)
    ax.invert_yaxis()

plot_grid(categorical_cols, 4, plot_categorical_bar, "Categorical Feature Distributions")

### 1.2.2 Skewness and Outlier Assessment

A few of the numeric features are visibly right-skewed. Here we quantify the skewness and use box plots to spot outliers. As a rule of thumb, skewness beyond +/- 1 is considered significant. For outliers, we use the IQR method (points beyond 1.5x the interquartile range).

In [ ]:
# Compute skewness using scipy and display sorted results
skew_records = [
    (col, round(stats.skew(df.get_column(col).to_list()), 2))
    for col in numeric_cols
]
skew_df = pl.DataFrame({"Feature": [r[0] for r in skew_records], "Skewness": [r[1] for r in skew_records]})
skew_df = skew_df.sort("Skewness", descending=True)

print("Skewness of numeric features (|skew| > 1 is notable):\n")
for feat, skew in skew_df.iter_rows():
    flag = " !!" if abs(skew) > 1 else ""
    print(f"  {feat:30s} {skew:>6.2f}{flag}")

In [ ]:
# Box plot + outlier count for each numeric feature
def plot_boxplot(ax, col):
    data = df.get_column(col).to_list()
    bp = ax.boxplot(data, vert=True, patch_artist=True, widths=0.5)
    bp["boxes"][0].set(facecolor="steelblue", alpha=0.7)
    ax.set_title(col, fontsize=11)
    ax.set_xticks([])

    # Count IQR outliers
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    n_outliers = sum(1 for v in data if v < q1 - 1.5 * iqr or v > q3 + 1.5 * iqr)
    if n_outliers > 0:
        ax.set_xlabel(f"{n_outliers} outliers", fontsize=9, color="red")

plot_grid(numeric_cols, 4, plot_boxplot, "Outlier Detection (IQR Method)")

### 1.2.3 Log Transform for Skewed Features

For features with skewness above 1, we apply a `log1p` transform (log(1 + x)) to pull in the right tail. We use `log1p` instead of plain `log` because some features contain zeros. This mainly benefits LASSO and other linear models; tree-based models like Random Forest won't be affected either way.

In [ ]:
# Apply log1p to features with |skewness| > 1
skewed_cols = [feat for feat, skew in skew_df.iter_rows() if abs(skew) > 1]
print(f"Applying log1p to {len(skewed_cols)} skewed features: {skewed_cols}\n")

for col in skewed_cols:
    df = df.with_columns(pl.col(col).cast(pl.Float64).log1p().alias(f"{col}_log"))

print(f"Added columns: {[f'{c}_log' for c in skewed_cols]}")
print(f"DataFrame shape: {df.shape[0]} rows, {df.shape[1]} columns")

# Side-by-side comparison: original vs transformed
fig, axes = plt.subplots(len(skewed_cols), 2, figsize=(14, 4 * len(skewed_cols)))

for i, col in enumerate(skewed_cols):
    axes[i][0].hist(df.get_column(col).to_list(), bins=30, edgecolor="white", alpha=0.8)
    axes[i][0].set_title(f"{col} (original)")

    axes[i][1].hist(df.get_column(f"{col}_log").to_list(), bins=30, edgecolor="white", alpha=0.8, color="orange")
    axes[i][1].set_title(f"{col} (log1p)")

fig.suptitle("Before vs After Log Transform", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 1.3.1 Target Variable Inspection

We look at the distribution of `Attrition` to understand how imbalanced our classes are. This matters because if one class heavily outweighs the other, our models could just predict the majority class and still look accurate on paper.

In [ ]:
# Attrition class distribution
counts = df["Attrition"].value_counts().sort("Attrition")
total = len(df)

fig, ax = plt.subplots(figsize=(6, 4))
labels = counts.get_column("Attrition").to_list()
values = counts.get_column("count").to_list()
bars = ax.bar(labels, values, color=["steelblue", "salmon"], edgecolor="white")

# Add count and percentage labels on each bar
for bar, val in zip(bars, values):
    pct = val / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f"{val} ({pct:.1f}%)", ha="center", fontsize=11)

ax.set_title("Attrition Class Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

# Print imbalance ratio
majority, minority = max(values), min(values)
print(f"Imbalance ratio: {majority / minority:.1f} : 1")

### 1.3.2 Encoding Categorical Variables

We convert string columns into numeric form so they can be fed into our models. Binary columns (`Attrition`, `OverTime`, `Gender`) get mapped to 0/1, while multi-class columns (`BusinessTravel`, `Department`, `EducationField`, `JobRole`, `MaritalStatus`) are one-hot encoded.

In [ ]:
# Binary encoding for two-value columns
binary_maps = {
    "Attrition": {"No": 0, "Yes": 1},
    "OverTime":  {"No": 0, "Yes": 1},
    "Gender":    {"Female": 0, "Male": 1},
}

for col, mapping in binary_maps.items():
    df = df.with_columns(pl.col(col).replace(mapping).cast(pl.Int64).alias(col))

print(f"Binary encoded: {list(binary_maps.keys())}")

# One-hot encoding for multi-class columns
multi_class_cols = ["BusinessTravel", "Department", "EducationField", "JobRole", "MaritalStatus"]
df = df.to_dummies(columns=multi_class_cols)

new_cols = [c for c in df.columns if any(c.startswith(f"{mc}_") for mc in multi_class_cols)]
print(f"One-hot encoded: {multi_class_cols} -> {len(new_cols)} new columns")
print(f"\nFinal shape: {df.shape[0]} rows, {df.shape[1]} columns")

### Part 1 Findings

**Data Quality**
- The dataset is clean out of the box: no missing values and no duplicate rows across all 1,470 records. Four columns (`EmployeeCount`, `StandardHours`, `Over18`, `EmployeeNumber`) were dropped because they were either constant or just row IDs, leaving 31 meaningful features.

**Distributions**
- Most numeric features are right-skewed. Five in particular (`YearsSinceLastPromotion`, `YearsAtCompany`, `MonthlyIncome`, `TotalWorkingYears`, `NumCompaniesWorked`) had skewness above 1.0, so we created `log1p` versions of these for use with linear models like LASSO.
- `DailyRate`, `HourlyRate`, and `MonthlyRate` are roughly uniformly distributed, which is unusual for salary-related fields.
- Ordinal satisfaction features (`JobSatisfaction`, `EnvironmentSatisfaction`, `RelationshipSatisfaction`) are fairly evenly spread across their four levels, meaning there's no dominant sentiment skewing the data.

**Outliers**
- Several tenure and income features have IQR outliers on the upper end. These represent senior, long-tenured employees rather than data errors, so we kept them in the dataset.

**Class Imbalance**
- Attrition is heavily imbalanced at roughly 5:1 (83.9% No vs 16.1% Yes). This will need to be addressed during modeling, either through oversampling to prevent the models from simply predicting "No" for everyone.

**After Encoding**
- Binary mapping was applied to `Attrition`, `OverTime`, and `Gender`. One-hot encoding expanded `BusinessTravel`, `Department`, `EducationField`, `JobRole`, and `MaritalStatus` into 24 dummy columns. The final dataset has 1,470 rows and 55 columns ready for modeling.

## Part 2: Feature Engineering

Before we build any models, we must engineer the necessary features. This section includes:

- Tenure calculations
- Satisfaction composites
- Identification of censored vs. uncensored observations

### 2.1 Tenure Calculations

The dataset has several raw tenure columns (`YearsAtCompany`, `TotalWorkingYears`, `YearsInCurrentRole`, `YearsWithCurrManager`, `YearsSinceLastPromotion`). This section will extract _ratios and rates_ that are more informative than the raw values alone.

In [ ]:
# + 1 in denominators to avoid division by zero for new hires
df = df.with_columns([
    # Fraction of total career spent at this company
    (pl.col('YearsAtCompany') / (pl.col('TotalWorkingYears') + 1)).alias('TenureCompanyRatio'),

    # Fraction of time at company spent in current role
    (pl.col('YearsInCurrentRole') / (pl.col('YearsAtCompany') + 1)).alias('TenureRoleRatio'),

    # Fraction of time at company under current manager
    (pl.col('YearsWithCurrManager') / (pl.col('YearsAtCompany') + 1)).alias('TenureManagerRatio'),

    #Average tenure per company (job-hopping signal)
    (pl.col('TotalWorkingYears') / (pl.col('NumCompaniesWorked') + 1)).alias('AvgTenurePerCompany'),

    # Promotion lag: years stagnant relative to total tenure
    (pl.col('YearsSinceLastPromotion') / (pl.col('YearsAtCompany') + 1)).alias('PromotionLagRatio'),

    #Career stage bins
    pl.col('TotalWorkingYears')
        .cut(breaks=[2, 5, 10, 20],
             labels=['entry', 'early', 'mid', 'senior', 'veteran'])
        .alias('CareerStage')
])

df.select(['TenureCompanyRatio', 'TenureRoleRatio', 'TenureManagerRatio', 'AvgTenurePerCompany', 'PromotionLagRatio', 'CareerStage']).head()

#### Key Points

- When `TenureCompanyRatio` is near 1.0, this indicates that the employee's entire career is at this company
- When `TenureRoleRatio` is near 0, this indicates that the employeed recently moved roles
- When `AvgTenurePerCompany` is low, this indicates that the employee is a job-hopper
- When `PromotionLagRatio` is high, this indicates that the employee's career may be stagnant

### 2.2 Satisfaction Composites

The dataset has 5 engagement and satisfaction columns, all on scales of 1 to 4: `JobSatisfaction`, `EnvironmentSatisfaction`, `RelationshipSatisfaction`, `WorkLifeBalance` and `JobInvolvement`. Composities reduce dimensionality and capture latent "overall engagement".

In [ ]:
SAT_COLS = [
    'JobSatisfaction',
    'EnvironmentSatisfaction',
    'RelationshipSatisfaction',
    'WorkLifeBalance',
    'JobInvolvement',
]

df = df.with_columns([
    # Simple mean (equal weight) — overall engagement score
    pl.concat_list([pl.col(c) for c in SAT_COLS])
        .list.mean()
        .alias('SatisfactionMean'),

    # Minimum — captures the "weakest link" dimension
    pl.concat_list([pl.col(c) for c in SAT_COLS])
        .list.min()
        .alias('SatisfactionMin'),

    # Standard deviation — measures inconsistency across dimensions
    # High variance = dissatisfied in some areas but not others
    pl.concat_list([pl.col(c).cast(pl.Float64) for c in SAT_COLS])
        .map_elements(lambda x: x.to_numpy().std(), return_dtype=pl.Float64)
        .alias('SatisfactionStd'),

    # Binary "at-risk" flag: mean below 2.5 (below midpoint of 1-4 scale)
    (pl.concat_list([pl.col(c) for c in SAT_COLS])
        .list.mean() < 2.5)
        .cast(pl.Int8)
        .alias('LowSatisfactionFlag'),

    # Work-life + job satisfaction sub-composite (most attrition-linked)
    ((pl.col('WorkLifeBalance') + pl.col('JobSatisfaction')) / 2)
        .alias('WLBJobComposite'),
])

df.select(['SatisfactionMean', 'SatisfactionMin', 'SatisfactionStd', 'LowSatisfactionFlag', 'WLBJobComposite']).head()

### Key Points

- High `SatisfactionMean` = Broadly engaged, Low `SatisfactionMean` = Broadly Disengaged
- High `SatisfactionMin` = No critical pain point, Low `SatisfactionMin` = At least one severe issue
- High `SatisfactionStd` = Uneven experience, Low `SatisfactionStd` = Consistently good or bad
- When `LowSatisfactionFlag == True`, satisfaction is low

### 2.3 Identifying Censored vs. Uncensored Observations

An uncensored observation is one where the employee left during the observation window (`Attrition == 1`) while a censored observation is one where the employee was _still employed_ when the data was collected (`Attrition == 0`). For censored observations, we don't know if/when employees will leave.

All `Attrition == 0`employees are **right-censored**: they haven't had the event yet, but may in the future. For survival analysis, we must encode this explicitly.

In [ ]:
df = df.with_columns([
    # Event indicator: 1 = uncensored (attrition observed), 0 = censored
    pl.col('Attrition').alias('EventObserved'),  # already 0/1 from encoding

    # Survival time = duration employee was observed
    # YearsAtCompany is the best proxy here (no hire/exit dates in this dataset)
    pl.col('YearsAtCompany').alias('SurvivalTime'),

    # Explicit censoring flag for clarity (inverse of event)
    (pl.col('Attrition') == 0).cast(pl.Int8).alias('IsCensored'),
])

# Summary
n_events   = df["EventObserved"].sum()
n_censored = df["IsCensored"].sum()
print(f"Uncensored (left): {n_events}  ({n_events/len(df)*100:.1f}%)")
print(f"Censored (stayed): {n_censored} ({n_censored/len(df)*100:.1f}%)")

### 2.4 Ordinal Encoding
There is one categorical engineered column: `CareerStage`. Ordinal encoding is used for `CareerStage` instead of one-hot encoding because the categories have a clear natural order: `entry < early < mid < senior < veteran` This means the values carry rank/experience progression, not just separate labels.

In [ ]:
# Ordinal encode CareerStage using a manual map to preserve the natural career progression order
CareerStageMap = {'entry': 0, 'early': 1, 'mid': 2, 'senior': 3, 'veteran': 4}
df = df.with_columns(
    pl.col('CareerStage').cast(pl.String).replace(CareerStageMap).cast(pl.Int64)
)

## Part 3: Classification

In this section, we apply different machine learning models to predict employee attrition.  
Because the dataset is imbalanced (many more "no attrition" cases than "attrition"), we use techniques like **class weighting** and **SMOTE** to ensure fair treatment of minority cases.  

We evaluate eight models:
- Logistic Regression with class weights  
- LASSO Logistic Regression (regularized) with class weights  
- Logistic Regression with SMOTE oversampling
- LASSO Logistic Regression with SMOTE oversampling
- Random Forest with class weights  
- Random Forest with SMOTE oversampling
- XGBoost with SMOTE oversampling
- XGBoost with class weights

Performance is assessed using accuracy, precision, recall, F1-score, AUROC, and confusion matrices.  


### 3.1 Train-Test Split

Splits the dataset into training and testing sets (80/20) while preserving class proportions.

In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split
import pandas as pd

# Drop target + leakage columns
leakage_cols = ["EventObserved", "SurvivalTime", "IsCensored"]
X = pd.DataFrame(
    df.drop(leakage_cols + ["Attrition"]).to_numpy(),
    columns=[c for c in df.columns if c not in leakage_cols + ["Attrition"]]
)
y = pd.Series(df["Attrition"].to_numpy(), name="Attrition")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")


### 3.2 SMOTE Oversampling

Balances the training data by generating synthetic minority samples to address class imbalance


In [ ]:
# SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"Original training set: {len(X_train)} samples, {y_train.sum()} positive")
print(f"SMOTE training set: {len(X_train_sm)} samples, {y_train_sm.sum()} positive")


### 3.3 Models
Fit and test multiple models (Logistic Regression, LASSO, Random Forest, XGBoost) on the balanced dataset to compare performance.

### 3.3.1 Logistic Regression with Class weights

Trains a logistic regression model that adjusts for class imbalance by weighting minority cases more heavily.

In [ ]:
# Logistic Regression with class_weight="balanced" to handle imbalance without oversampling
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression with more iterations
log_reg = LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:,1]

print("Logistic Regression (weights)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))



### 3.3.2 Lasso Logistic Regression with Class Weights
Applies L1-regularized logistic regression with class weighting to handle imbalance and perform feature selection.

In [ ]:
# Lasso Logistic Regression for feature selection and weight
lasso = LogisticRegression(
    l1_ratio=1.0,           # 1.0 = pure L1
    solver='saga',          # supports elasticnet
    class_weight='balanced',
    max_iter=5000,
    random_state=42
)
lasso.fit(X_train_scaled, y_train)

y_pred = lasso.predict(X_test_scaled)
y_prob = lasso.predict_proba(X_test_scaled)[:,1]

print("LASSO Logistic Regression (weights)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))

### 3.3.3 Logistic Regression with SMOTE
Trains a logistic regression model on SMOTE-oversampled data to handle class imbalance through synthetic minority sampling rather than class weighting.


In [ ]:
# Logistic Regression trained on SMOTE-oversampled data (no class weights needed since classes are balanced)
scaler_sm = StandardScaler()
X_train_sm_scaled = scaler_sm.fit_transform(X_train_sm)
X_test_sm_scaled  = scaler_sm.transform(X_test)

log_reg_smote = LogisticRegression(max_iter=5000, random_state=42)
log_reg_smote.fit(X_train_sm_scaled, y_train_sm)

y_pred = log_reg_smote.predict(X_test_sm_scaled)
y_prob = log_reg_smote.predict_proba(X_test_sm_scaled)[:, 1]

print("Logistic Regression (SMOTE)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))


### 3.3.4 LASSO Logistic Regression with SMOTE
Applies L1-regularized logistic regression on SMOTE-oversampled data, combining feature selection with synthetic minority sampling to handle class imbalance.

In [ ]:
# LASSO Logistic Regression trained on SMOTE-oversampled data
lasso_smote = LogisticRegression(
    l1_ratio=1.0,           # 1.0 = pure L1
    solver='saga',          # supports elasticnet
    max_iter=5000,
    random_state=42
)
lasso_smote.fit(X_train_sm_scaled, y_train_sm)

y_pred = lasso_smote.predict(X_test_sm_scaled)
y_prob = lasso_smote.predict_proba(X_test_sm_scaled)[:, 1]

print("LASSO Logistic Regression (SMOTE)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))

### 3.3.5 Random Forest with Class Weights
Trains a random forest model that compensates for class imbalance by weighting minority cases more heavily.

In [ ]:
#Random Forest + Weights
from sklearn.ensemble import RandomForestClassifier

rf_weights = RandomForestClassifier(class_weight="balanced", random_state=42)
rf_weights.fit(X_train, y_train)

y_pred = rf_weights.predict(X_test)
y_prob = rf_weights.predict_proba(X_test)[:,1]

print("Random Forest (weights)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))


### 3.3.6 Random Forest with SMOTE
Trains a random forest model on a SMOTE-balanced dataset to improve recall for minority cases.

In [ ]:
# Random Forest trained on SMOTE-oversampled data (no class weights needed since classes are balanced)

rf_smote = RandomForestClassifier(random_state=42)
rf_smote.fit(X_train_sm, y_train_sm)

y_pred = rf_smote.predict(X_test)
y_prob = rf_smote.predict_proba(X_test)[:,1]

print("Random Forest (SMOTE)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))



### 3.3.7 XGBoost with SMOTE
Trains a XGBoost model on a SMOTE-balanced dataset to improve recall for minority cases.

In [ ]:
# XGBoost trained on SMOTE-oversampled data (no class weights needed since classes are balanced)
from xgboost import XGBClassifier

xgb_smote = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb_smote.fit(X_train_sm, y_train_sm)

y_pred = xgb_smote.predict(X_test)
y_prob = xgb_smote.predict_proba(X_test)[:, 1]

print("XGBoost (SMOTE)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))


### 3.3.8 XGBoost with Class Weights

Trains a XGBoost model to account for class imbalance by weighting minority classes more heavily.

In [ ]:
#XGBoost + weights
from sklearn.utils.class_weight import compute_sample_weight

# compute class weights then convert into sample weights
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_weighted = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_weighted.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

y_pred = xgb_weighted.predict(X_test)
y_prob = xgb_weighted.predict_proba(X_test)[:, 1]

print("XGBoost (Class Weights)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("AUROC:", roc_auc_score(y_test, y_prob))

### 3.4 Evaluation
Compares precision, recall, F1-score, and accuracy across models using the test set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    print(f"\n{name}")
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

# Logistic Regression + LASSO → scaled test set
evaluate_model(log_reg, X_test_scaled, y_test, "Logistic Regression + Weights")
evaluate_model(lasso, X_test_scaled, y_test, "LASSO Logistic Regression + Weights")

# Logistic Regression + LASSO → SMOTE-scaled test set
evaluate_model(log_reg_smote, X_test_sm_scaled, y_test, "Logistic Regression + SMOTE")
evaluate_model(lasso_smote, X_test_sm_scaled, y_test, "LASSO Logistic Regression + SMOTE")

# Random Forest → raw test set
evaluate_model(rf_weights, X_test, y_test, "Random Forest + Weights")
evaluate_model(rf_smote, X_test, y_test, "Random Forest + SMOTE")

# XGBoost → raw test set
evaluate_model(xgb_smote, X_test, y_test, "XGBoost + SMOTE")
evaluate_model(xgb_weighted, X_test, y_test, "XGBoost + Weights")

### 3.4.1 Confusion Matrix Heatmaps
Visualizes each model’s predictions with confusion matrices to compare classification performance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

models = [
    ("Logistic Regression + Weights", log_reg, X_test_scaled),
    ("LASSO Logistic Regression + Weights", lasso, X_test_scaled),
    ("Logistic Regression + SMOTE", log_reg_smote, X_test_sm_scaled),
    ("LASSO Logistic Regression + SMOTE", lasso_smote, X_test_sm_scaled),
    ("Random Forest + Weights", rf_weights, X_test),
    ("Random Forest + SMOTE", rf_smote, X_test),
    ("XGBoost + SMOTE", xgb_smote, X_test),
    ("XGBoost + Weights", xgb_weighted, X_test)
]

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.ravel()

for ax, (name, model, X_eval) in zip(axes, models):
    y_pred = model.predict(X_eval)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

### 3.4.2 Model Comparison Summary

| Model | Accuracy | Precision | Recall | F1 | AUROC |
|-------|----------|-----------|--------|-----|-------|
| Logistic Regression (weights) | 0.762 | 0.358 | 0.617 | 0.453 | 0.787 |
| LASSO Logistic Regression (weights) | 0.769 | 0.367 | 0.617 | 0.460 | 0.792 |
| Logistic Regression + SMOTE | 0.772 | 0.368 | 0.596 | 0.455 | 0.789 |
| LASSO Logistic Regression + SMOTE | 0.772 | 0.368 | 0.596 | 0.455 | 0.794 |
| Random Forest (weights) | 0.833 | 0.400 | 0.085 | 0.140 | 0.766 |
| Random Forest (SMOTE) | 0.854 | 0.625 | 0.213 | 0.317 | 0.779 |
| XGBoost (SMOTE) | 0.857 | 0.647 | 0.234 | 0.344 | 0.782 |
| XGBoost (weights) | 0.837 | 0.487 | 0.404 | 0.442 | 0.771 |

**Key Observations:**

**Logistic Regression variants** achieve the highest recall (0.60–0.62), catching the most attrition cases, but with lower precision (~0.36–0.37). The class-weighted versions slightly outperform SMOTE on recall (0.62 vs 0.60), while SMOTE produces marginally higher accuracy. LASSO edges out standard LR in both cases with a slightly higher AUROC (0.792/0.794 vs 0.787/0.789), confirming that L1 regularization adds value through feature selection.

**Random Forest** models have high accuracy (0.83–0.85) but critically low recall (0.09–0.21), meaning they miss most attrition cases. SMOTE more than doubles RF recall (0.09 → 0.21) but it remains far too low for practical use.

**XGBoost (weights)** stands out as the best tree-based option. It achieves the highest recall among tree models (0.40) with reasonable precision (0.49), giving it the strongest F1 score (0.44) after the logistic regression variants. Class weighting works noticeably better than SMOTE for XGBoost on this dataset.

**XGBoost (SMOTE)** has the highest overall accuracy (0.86) and precision (0.65), but recall drops to 0.23 — it's conservative, predicting attrition only when very confident.

**Conclusion**: For attrition prediction where missing a flight-risk employee is costly, **LASSO Logistic Regression** (either with weights or SMOTE) is the recommended model — it achieves the highest recall and AUROC while providing interpretable feature selection. **XGBoost with class weights** is the strongest tree-based alternative, offering a better balance of precision and recall than any other ensemble model. If minimising false alarms is the priority, **XGBoost + SMOTE** provides the highest precision. Random Forest models are unsuitable for this task regardless of balancing technique.

### 3.5 Cross-Validation

So far, all our results come from a single 80/20 train-test split. With only 47 attrition cases in the test set, that's a small sample to draw conclusions from. A few employees landing in one fold versus another can swing recall by 10+ percentage points.

To get more reliable estimates, we run stratified 5-fold cross-validation on each model. This trains and evaluates each model five times on different splits, preserving class proportions each time. The mean and standard deviation across folds tell us not just how well each model performs, but how *stable* that performance is.

In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# Define pipelines so SMOTE is applied within each fold (no data leakage)
pipelines = {
    'LR (weights)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42))
    ]),
    'LASSO (weights)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(l1_ratio=1.0, solver='saga', class_weight='balanced', max_iter=5000, random_state=42))
    ]),
    'LR + SMOTE': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=5000, random_state=42))
    ]),
    'LASSO + SMOTE': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(l1_ratio=1.0, solver='saga', max_iter=5000, random_state=42))
    ]),
    'RF (weights)': Pipeline([
        ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
    ]),
    'RF + SMOTE': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', RandomForestClassifier(random_state=42))
    ]),
    'XGB + SMOTE': ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('clf', XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, eval_metric='logloss', random_state=42))
    ]),
    'XGB (weights)': Pipeline([
        ('clf', XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, eval_metric='logloss', random_state=42))
    ]),
}

# For XGB (weights), we need sample weights — handle via fit_params in cross_validate
# But cross_validate doesn't support per-fold sample weights easily, so we use a workaround:
# train it without sample_weight in CV (using scale_pos_weight instead)
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y)
scale_pos = class_weights[1] / class_weights[0]

pipelines['XGB (weights)'] = Pipeline([
    ('clf', XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, eval_metric='logloss', scale_pos_weight=scale_pos,
                          random_state=42))
])

# Run cross-validation
cv_results = {}
for name, pipe in pipelines.items():
    results = cross_validate(pipe, X, y, cv=cv, scoring=scoring, return_train_score=False)
    cv_results[name] = results
    print(f"{name}:")
    for metric in scoring:
        scores = results[f'test_{metric}']
        print(f"  {metric:>10s}: {scores.mean():.3f} (+/- {scores.std():.3f})")
    print()

In [ ]:
# Cross-validation summary table
cv_summary = pd.DataFrame({
    name: {
        metric: f"{results[f'test_{metric}'].mean():.3f} \u00b1 {results[f'test_{metric}'].std():.3f}"
        for metric in scoring
    }
    for name, results in cv_results.items()
}).T

cv_summary.columns = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUROC']
cv_summary.index.name = 'Model'
print(cv_summary.to_string())

### 3.6 Hyperparameter Tuning

Our models so far use either default hyperparameters (Random Forest) or hand-picked ones (XGBoost). There's likely performance left on the table, especially for Random Forest, which is currently underperforming.

We use `RandomizedSearchCV` with stratified 5-fold CV to search for better hyperparameters. We optimise for **recall** since our primary goal is catching attrition cases, but we also track F1 and AUROC to make sure we're not sacrificing too much precision.

We tune four models:
- **LASSO Logistic Regression (weights)**: tuning regularisation strength (C)
- **Random Forest (SMOTE)**: tuning tree depth, number of estimators, and split criteria
- **XGBoost (SMOTE)**: tuning learning rate, depth, and regularisation
- **XGBoost (weights)**: same search space, different balancing strategy

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- LASSO (weights) ---
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(l1_ratio=1.0, solver='saga', class_weight='balanced', max_iter=5000, random_state=42))
])
lasso_params = {'clf__C': uniform(0.01, 10)}

lasso_search = RandomizedSearchCV(
    lasso_pipe, lasso_params, n_iter=50, scoring='recall', cv=cv_tune, random_state=42, n_jobs=-1
)
lasso_search.fit(X_train, y_train)
print("LASSO (weights) best params:", lasso_search.best_params_)
print(f"  Best CV recall: {lasso_search.best_score_:.3f}")
print()

In [ ]:
# --- Random Forest (SMOTE) ---
rf_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(random_state=42))
])
rf_params = {
    'clf__n_estimators': randint(100, 500),
    'clf__max_depth': [3, 5, 8, 12, 20, None],
    'clf__min_samples_split': randint(2, 20),
    'clf__min_samples_leaf': randint(1, 10),
    'clf__max_features': ['sqrt', 'log2', None]
}

rf_search = RandomizedSearchCV(
    rf_pipe, rf_params, n_iter=50, scoring='recall', cv=cv_tune, random_state=42, n_jobs=-1
)
rf_search.fit(X_train, y_train)
print("Random Forest (SMOTE) best params:", rf_search.best_params_)
print(f"  Best CV recall: {rf_search.best_score_:.3f}")
print()

In [ ]:
# --- XGBoost (SMOTE) ---
xgb_smote_pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', XGBClassifier(eval_metric='logloss', random_state=42))
])
xgb_params = {
    'clf__n_estimators': randint(100, 500),
    'clf__max_depth': randint(3, 10),
    'clf__learning_rate': uniform(0.01, 0.3),
    'clf__subsample': uniform(0.6, 0.4),
    'clf__colsample_bytree': uniform(0.6, 0.4),
    'clf__reg_alpha': uniform(0, 1),
    'clf__reg_lambda': uniform(0.5, 2)
}

xgb_smote_search = RandomizedSearchCV(
    xgb_smote_pipe, xgb_params, n_iter=50, scoring='recall', cv=cv_tune, random_state=42, n_jobs=-1
)
xgb_smote_search.fit(X_train, y_train)
print("XGBoost (SMOTE) best params:", xgb_smote_search.best_params_)
print(f"  Best CV recall: {xgb_smote_search.best_score_:.3f}")
print()

In [ ]:
# --- XGBoost (weights) ---
xgb_w_pipe = Pipeline([
    ('clf', XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos, random_state=42))
])
xgb_w_params = {
    'clf__n_estimators': randint(100, 500),
    'clf__max_depth': randint(3, 10),
    'clf__learning_rate': uniform(0.01, 0.3),
    'clf__subsample': uniform(0.6, 0.4),
    'clf__colsample_bytree': uniform(0.6, 0.4),
    'clf__reg_alpha': uniform(0, 1),
    'clf__reg_lambda': uniform(0.5, 2)
}

xgb_w_search = RandomizedSearchCV(
    xgb_w_pipe, xgb_w_params, n_iter=50, scoring='recall', cv=cv_tune, random_state=42, n_jobs=-1
)
xgb_w_search.fit(X_train, y_train)
print("XGBoost (weights) best params:", xgb_w_search.best_params_)
print(f"  Best CV recall: {xgb_w_search.best_score_:.3f}")
print()

### 3.6.1 Tuned Model Evaluation
Now we evaluate the tuned models on the held-out test set and compare them against the original (untuned) results.

In [ ]:
# Evaluate tuned models on test set
tuned_models = {
    'LASSO (weights) - tuned': (lasso_search.best_estimator_, X_test),
    'RF + SMOTE - tuned': (rf_search.best_estimator_, X_test),
    'XGB + SMOTE - tuned': (xgb_smote_search.best_estimator_, X_test),
    'XGB (weights) - tuned': (xgb_w_search.best_estimator_, X_test),
}

print("Tuned Model Results on Test Set")
print("=" * 65)

tuned_results = []
for name, (model, X_eval) in tuned_models.items():
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auroc = roc_auc_score(y_test, y_prob)
    
    tuned_results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'AUROC': auroc})
    
    print(f"\n{name}")
    print(f"  Accuracy:  {acc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall:    {rec:.3f}")
    print(f"  F1:        {f1:.3f}")
    print(f"  AUROC:     {auroc:.3f}")

tuned_df = pd.DataFrame(tuned_results)
print("\n" + tuned_df.to_string(index=False))

In [ ]:
# Side-by-side: original vs tuned
original = [
    ('LASSO (weights)', lasso, X_test_scaled),
    ('RF + SMOTE', rf_smote, X_test),
    ('XGB + SMOTE', xgb_smote, X_test),
    ('XGB (weights)', xgb_weighted, X_test),
]

tuned = [
    ('LASSO (weights) - tuned', lasso_search.best_estimator_, X_test),
    ('RF + SMOTE - tuned', rf_search.best_estimator_, X_test),
    ('XGB + SMOTE - tuned', xgb_smote_search.best_estimator_, X_test),
    ('XGB (weights) - tuned', xgb_w_search.best_estimator_, X_test),
]

print(f"{'Model':<30s} {'Recall':>8s} {'F1':>8s} {'AUROC':>8s}")
print("-" * 58)

for (orig_name, orig_model, orig_X), (tuned_name, tuned_model, tuned_X) in zip(original, tuned):
    # Original
    y_pred_o = orig_model.predict(orig_X)
    y_prob_o = orig_model.predict_proba(orig_X)[:, 1]
    rec_o = recall_score(y_test, y_pred_o)
    f1_o = f1_score(y_test, y_pred_o)
    auc_o = roc_auc_score(y_test, y_prob_o)
    
    # Tuned
    y_pred_t = tuned_model.predict(tuned_X)
    y_prob_t = tuned_model.predict_proba(tuned_X)[:, 1]
    rec_t = recall_score(y_test, y_pred_t)
    f1_t = f1_score(y_test, y_pred_t)
    auc_t = roc_auc_score(y_test, y_prob_t)
    
    rec_delta = rec_t - rec_o
    f1_delta = f1_t - f1_o
    auc_delta = auc_t - auc_o
    
    print(f"{orig_name:<30s} {rec_o:>8.3f} {f1_o:>8.3f} {auc_o:>8.3f}")
    print(f"{tuned_name:<30s} {rec_t:>8.3f} {f1_t:>8.3f} {auc_t:>8.3f}  ({rec_delta:+.3f} / {f1_delta:+.3f} / {auc_delta:+.3f})")
    print()

### 3.6.2 Tuning Findings

**Cross-validation confirms what the single split suggested, but with more nuance:**

The Logistic Regression variants are the most *stable* models. LASSO (weights) achieves 0.730 recall with a tight standard deviation of just 0.022, meaning it performs consistently across different data splits, not just on one lucky test set.

XGBoost (weights) has the highest F1 in CV (0.515) but also the highest variance (0.056), reflecting a trade-off between capturing more attrition cases and being less predictable about it.

Random Forest (weights) is confirmed as the weakest model: 0.143 recall with a wide spread of 0.063. It's both inaccurate and unreliable.

**Hyperparameter tuning made the biggest difference where it was needed most:**

**Random Forest (SMOTE)** saw the largest improvement: recall jumped from 0.213 to 0.468 (+0.255) and F1 from 0.317 to 0.473 (+0.156). The tuned model uses shallow trees (max_depth=3) and more conservative splits, which prevents it from memorising the majority class. This confirms that RF's poor baseline performance was largely a hyperparameter issue, not a fundamental limitation.

**XGBoost (SMOTE)** also improved substantially: recall from 0.234 to 0.362 (+0.128) and F1 from 0.344 to 0.486 (+0.142). The tuned version uses a higher learning rate (0.22 vs 0.05) and more regularisation.

**LASSO (weights)** was essentially unchanged by tuning. The optimal C (0.59) is close to the default (1.0), confirming that the linear model was already well-configured.

**XGBoost (weights)** held steady on recall (0.404) but gained AUROC (+0.028), suggesting better probability calibration even if the hard predictions didn't change.

**Updated picture:**

After tuning, the model landscape shifts. LASSO (weights) still leads on recall (0.617), but the gap has narrowed. Tuned RF + SMOTE (0.468) and tuned XGBoost + SMOTE (0.362) are now viable options where they weren't before. For a production system, the choice depends on the cost trade-off: LASSO catches the most flight risks but generates more false alarms, while tuned XGBoost + SMOTE is more selective but misses more.

### 3.7 Model Recommendation

After training eight models, validating with cross-validation, and tuning hyperparameters, the choice of model comes down to what the organisation values most.

| Priority | Recommended Model | Why |
|----------|------------------|-----|
| Catch every flight risk | LASSO (weights) | Highest recall (0.73 CV), most stable across folds, interpretable coefficients |
| Balance catching and accuracy | XGBoost (weights) | Best F1 in CV (0.515), decent recall (0.47) with better precision (0.58) |
| Minimise false alarms | Tuned XGBoost + SMOTE | Highest precision (0.74). When it flags someone, it's usually right |

**Our recommendation: use LASSO (weights) as the primary model and XGBoost (weights) as a complementary signal.**

A two-model approach makes sense for this problem. LASSO casts a wide net: with 0.73 recall in cross-validation, it catches roughly three out of four employees who will actually leave. The trade-off is that about 60% of its flags are false alarms, but in a retention context, a false alarm just means checking in with an employee who wasn't planning to leave. That's a low-cost action.

XGBoost adds confidence scoring. It captures non-linear patterns that LASSO misses (satisfaction interactions, stock options, travel frequency). An employee flagged by *both* models is at very high risk and should trigger a direct intervention: a compensation review, a role change conversation, or a workload adjustment.

The combination gives you a tiered response. Flagged by LASSO only? Schedule a casual check-in. Flagged by both? Escalate to their manager and HR. Flagged by neither? Low risk for now, but keep monitoring.

This approach maximises coverage while giving the organisation a way to prioritise where to spend its retention budget. It also provides two different lenses on the problem: LASSO tells you *which factors* are driving risk through its coefficients, while XGBoost's SHAP values reveal *how factors interact* for each individual employee.

## Part 4: Feature Importance Analysis (SHAP)

Now that we know *how well* our models predict attrition, the next question is *why* they make the predictions they do. Which factors actually push an employee toward leaving?

To answer this, we use SHAP (SHapley Additive exPlanations), a method rooted in cooperative game theory that breaks down each prediction into individual feature contributions (Lundberg & Lee, 2017). The key advantage of SHAP over simpler importance measures is consistency: it gives reliable explanations regardless of the model type, so we can fairly compare what different models think matters.

We run SHAP on five models that span our linear and tree-based approaches:
- **LASSO Logistic Regression (weights)**: our best linear model, which also performs feature selection
- **LASSO Logistic Regression (SMOTE)**: same architecture but trained on SMOTE-balanced data, letting us see if the balancing strategy changes what the model focuses on
- **Random Forest (SMOTE)**: a tree ensemble that can pick up on non-linear patterns
- **XGBoost (SMOTE)**: gradient boosting trained on SMOTE-balanced data
- **XGBoost (weights)**: gradient boosting with class weighting, letting us compare balancing strategies for tree models too

By comparing explanations across these five, we can separate the features that consistently drive attrition from those that only matter to certain model types.

*Reference: Lundberg, S. M., & Lee, S. I. (2017). A Unified Approach to Interpreting Model Predictions. Advances in Neural Information Processing Systems, 30. https://proceedings.neurips.cc/paper/2017/hash/8a20a8621978632d76c43dfd28b67767-Abstract.html*

### 4.1 SHAP: LASSO Logistic Regression (Weights)
Since LASSO is a linear model, SHAP can derive each feature's contribution directly from the coefficients and input values. The plot below shows the top 15 features. Each dot is one employee, coloured by whether their feature value was high (red) or low (blue), and positioned by how much it pushed the prediction toward or away from attrition.

In [ ]:
import shap

# Linear explainer for LASSO with class weights (uses scaled data)
explainer_lasso = shap.LinearExplainer(lasso, X_train_scaled)
shap_values_lasso = explainer_lasso.shap_values(X_test_scaled)

# Summary plot — global feature importance with direction of effect
shap.summary_plot(shap_values_lasso, X_test, max_display=15, show=False)
plt.title("SHAP — LASSO Logistic Regression (Weights)")
plt.tight_layout()
plt.show()

### 4.2 SHAP: LASSO Logistic Regression (SMOTE)
This is the same LASSO architecture but trained on SMOTE-oversampled data instead of using class weights. Comparing this plot to 4.1 tells us whether the way we handle class imbalance changes which features the model leans on.

In [ ]:
# Linear explainer for LASSO with SMOTE (uses SMOTE-scaled data)
explainer_lasso_smote = shap.LinearExplainer(lasso_smote, X_train_sm_scaled)
shap_values_lasso_smote = explainer_lasso_smote.shap_values(X_test_sm_scaled)

# Summary plot
shap.summary_plot(shap_values_lasso_smote, X_test, max_display=15, show=False)
plt.title("SHAP — LASSO Logistic Regression (SMOTE)")
plt.tight_layout()
plt.show()

### 4.3 SHAP: Random Forest (SMOTE)
Random Forest can capture interactions between features that linear models miss. The tree-based SHAP explainer computes exact contribution values, so we get a fair comparison with the LASSO plots above.

In [ ]:
# Tree explainer for Random Forest
explainer_rf = shap.TreeExplainer(rf_smote)
shap_values_rf = explainer_rf.shap_values(X_test)

# For binary classification, use the positive class (index 1)
shap_values_rf_pos = shap_values_rf[:, :, 1] if shap_values_rf.ndim == 3 else shap_values_rf

shap.summary_plot(shap_values_rf_pos, X_test, max_display=15, show=False)
plt.title("SHAP — Random Forest (SMOTE)")
plt.tight_layout()
plt.show()

### 4.4 SHAP: XGBoost (SMOTE)
XGBoost builds trees sequentially, with each one correcting the mistakes of the last. This often surfaces different feature priorities than Random Forest, particularly features whose effect depends on the values of other variables.

In [ ]:
# Tree explainer for XGBoost (SMOTE)
explainer_xgb = shap.TreeExplainer(xgb_smote)
shap_values_xgb = explainer_xgb.shap_values(X_test)

shap.summary_plot(shap_values_xgb, X_test, max_display=15, show=False)
plt.title("SHAP — XGBoost (SMOTE)")
plt.tight_layout()
plt.show()

### 4.5 SHAP: XGBoost (Weights)
Same XGBoost architecture but trained with class weights instead of SMOTE. Comparing this to 4.4 shows whether the balancing strategy matters for tree-based models the same way it does for linear ones.

In [ ]:
# Tree explainer for XGBoost (Weights)
explainer_xgb_w = shap.TreeExplainer(xgb_weighted)
shap_values_xgb_w = explainer_xgb_w.shap_values(X_test)

shap.summary_plot(shap_values_xgb_w, X_test, max_display=15, show=False)
plt.title("SHAP — XGBoost (Weights)")
plt.tight_layout()
plt.show()

### 4.6 Cross-Model Feature Importance Comparison
To cut through the noise of individual model explanations, this chart puts all five models side by side. Features that show up as important across multiple models are the ones we can be most confident about. They represent genuine attrition signals rather than quirks of a particular algorithm.

In [ ]:
import numpy as np

# Compute mean absolute SHAP values for each model
feature_names = X_test.columns.tolist()

mean_shap_lasso_w = np.abs(shap_values_lasso).mean(axis=0)
mean_shap_lasso_sm = np.abs(shap_values_lasso_smote).mean(axis=0)
mean_shap_rf = np.abs(shap_values_rf_pos).mean(axis=0)
mean_shap_xgb = np.abs(shap_values_xgb).mean(axis=0)
mean_shap_xgb_w = np.abs(shap_values_xgb_w).mean(axis=0)

# Build comparison DataFrame
shap_comparison = pd.DataFrame({
    'Feature': feature_names,
    'LASSO (Weights)': mean_shap_lasso_w,
    'LASSO (SMOTE)': mean_shap_lasso_sm,
    'Random Forest (SMOTE)': mean_shap_rf,
    'XGBoost (SMOTE)': mean_shap_xgb,
    'XGBoost (Weights)': mean_shap_xgb_w
})

# Rank features by average importance across models
model_cols = ['LASSO (Weights)', 'LASSO (SMOTE)', 'Random Forest (SMOTE)', 'XGBoost (SMOTE)', 'XGBoost (Weights)']
shap_comparison['Mean Importance'] = shap_comparison[model_cols].mean(axis=1)
shap_comparison = shap_comparison.sort_values('Mean Importance', ascending=False)

# Plot top 15 features
top_n = 15
top_features = shap_comparison.head(top_n)

fig, ax = plt.subplots(figsize=(16, 7))
x = np.arange(top_n)
width = 0.15

ax.bar(x - 2 * width, top_features['LASSO (Weights)'], width, label='LASSO (Weights)')
ax.bar(x - width, top_features['LASSO (SMOTE)'], width, label='LASSO (SMOTE)')
ax.bar(x, top_features['Random Forest (SMOTE)'], width, label='Random Forest (SMOTE)')
ax.bar(x + width, top_features['XGBoost (SMOTE)'], width, label='XGBoost (SMOTE)')
ax.bar(x + 2 * width, top_features['XGBoost (Weights)'], width, label='XGBoost (Weights)')

ax.set_xticks(x)
ax.set_xticklabels(top_features['Feature'], rotation=45, ha='right')
ax.set_ylabel('Mean |SHAP Value|')
ax.set_title('Top 15 Features by Mean Absolute SHAP Value')
ax.legend()
plt.tight_layout()
plt.show()

### 4.7 SHAP Findings

**The big three: features every model agrees on**

**Monthly income is the strongest signal in the linear models.** Both LASSO variants place MonthlyIncome (and its log transform) at the top of the feature rankings by a wide margin. Lower pay clearly and consistently pushes employees toward the exit. It's the most straightforward lever the organisation can pull.

**Overtime is the top driver for tree models, and still ranks highly everywhere.**  Both XGBoost variants flag OverTime as their single most important feature. The LASSO models rank it among their top features as well. Random Forest assigns it a much lower weight, consistent with how RF distributes importance more evenly across features. Still, the message from the majority of models is consistent: chronic overwork is a major flight risk signal.

**Tenure and career progression matter across all model types.** TotalWorkingYears_log and TenureRoleRatio appear prominently across both linear and tree models. Employees who've been in the workforce longer but haven't progressed, or who've been stuck in the same role, are at elevated risk.

**What the tree models see that the linear models don't**

The LASSO models focus on clean, directional drivers: income, overtime, total working years, tenure ratios, and the low satisfaction flag. The relationships are straightforward. More overtime means more risk; higher income means less risk.

Random Forest surfaces **LowSatisfactionFlag** and **MaritalStatus_Single** as its #2 and #3 features. These are much less prominent in the linear models, suggesting their effect on attrition depends on interactions with other variables.

XGBoost picks up on **BusinessTravel_Travel_Frequently**, **DistanceFromHome**, and **StockOptionLevel** as important. These are contextual factors that likely compound risk when combined with other stressors like overtime or low pay.

**SatisfactionMin** and **SatisfactionMean** appear across multiple tree models, confirming that the satisfaction composites engineered in Part 2 capture real signal that individual satisfaction columns miss.

**Does the balancing strategy matter?**

Not much, which is reassuring. Both LASSO variants agree on the same top features in largely the same order. The two XGBoost variants show more variation: XGBoost (weights) places **AvgTenurePerCompany** and **SatisfactionMean** higher, while XGBoost (SMOTE) emphasises **BusinessTravel_Travel_Frequently** and **DistanceFromHome**. But their top 3 features (OverTime, LowSatisfactionFlag/StockOptionLevel) are consistent. The core findings are robust.

**The bottom line**

The SHAP analysis points to a clear, actionable retention strategy:

1. **Review compensation first.** It's the most consistent driver across model types. Employees in the lower income brackets are at significantly higher flight risk, and even modest adjustments could move the needle.
2. **Tackle overtime.** It's the single biggest lever in the tree models and ranks highly everywhere. Audit which teams carry the heaviest overtime burden and look for structural fixes, whether that's hiring, redistributing work, or setting boundaries.
3. **Watch for compounding risk factors.** The tree models tell us that attrition risk isn't just about one thing. It's about combinations. An employee who travels frequently, lives far from the office, *and* works overtime is at much higher risk than any of those factors alone would suggest.
4. **Create career movement.** Employees with long tenure but limited progression need a pathway forward. Stagnation doesn't always show up as dissatisfaction in the moment, but it makes people quietly open to leaving.

The fact that both our simple linear models and our more complex tree models converge on these core factors gives us confidence that these are real, actionable signals, not statistical noise.

# Part 5: Survival Analysis
## When Will Employees Leave — and Are Our Predictions Fair?

---

### Motivation :

Survival analysis asks not just *whether* an employee will leave, but ***when***. This temporal dimension makes it far more actionable for HR than a binary attrition classifier — it tells us how long we can expect an employee to stay, and which factors accelerate or delay their departure.

But predicting employee tenure and flagging flight-risk individuals carries real consequences. If a model systematically assigns higher attrition risk to employees in a protected group — women, older workers, a particular racial category — those employees may be denied development opportunities, excluded from high-visibility projects, or informally pushed out. We build the survival models **and** audit them at every stage for signs of discriminatory behaviour.

### Fairness Audit in Survival Analysis :

A fairness audit examines whether a predictive model treats different demographic groups equitably. In the survival context this means checking:

1. **Survival disparity** — do KM curves differ significantly across protected groups (Gender, Age bracket)?  
   Disparity in raw data is not itself a model fault — it may reflect real structural differences. But it alerts us to where the model will inherit and potentially amplify gaps.

2. **Demographic parity of risk scores** — does the distribution of predicted 1-year attrition probability differ systematically by group? A model achieving parity assigns similar proportions of each group to each risk tier.

3. **Calibration by subgroup** — within each risk tier, does actual attrition match predicted probability equally well for all groups? Poor calibration for a specific group signals the model is miscalibrated for them, leading to unfair resource allocation.

4. **Protected-attribute coefficients in Cox** — does the model directly encode protected attributes (Gender, Age) as risk factors with large hazard ratios? If so, predictions are partly driven by who an employee *is* rather than what their working conditions are.

In [ ]:
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.utils import concordance_index

# Color Palette for graphs
PALETTE = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0', '#00BCD4']
FAIR_PAL = {'Female': '#9C27B0', 'Male': '#2196F3','Young': '#4CAF50',  'Mid': '#FF9800', 'Senior': '#F44336'}

survival_df = df.with_columns([df['SurvivalTime'], df['EventObserved']])

n_survtimearr = survival_df['SurvivalTime'].to_numpy()
n_eventsarr = survival_df['EventObserved'].to_numpy()

print(f'Dataset ready: {len(survival_df)} employees')
print(f'  Left (events): {n_eventsarr.sum()} ({n_eventsarr.mean()*100:.1f}%)')
print(f'  Stayed (censored): {n_censored} ({n_censored/len(df)*100:.1f}%)')
print(f'  Survival time range: {n_survtimearr.min():.1f} – {n_survtimearr.max():.1f} years')

---
## 5.1 Kaplan-Meier Survival Estimator

The Kaplan-Meier estimator answers a deceptively simple question: *What is the probability that an employee hired today is still here after t years?* It does so non-parametrically, requiring no distributional assumptions, while correctly handling **censored observations** — employees who were still employed when the data was collected.

### Formula :

$$\hat{S}(t) = \prod_{t_i \leq t} \left(1 - \frac{d_i}{n_i}\right)$$

- $t_i$ — each observed departure time  
- $d_i$ — number of employees who left at exactly $t_i$  
- $n_i$ — number still employed (and uncensored) just before $t_i$

### What we will compute :

1. The **overall** KM curve for the full population with 95% confidence bands  
2. **Stratified** curves by key operational factors (overtime, satisfaction, job level) to identify retention levers  
3. **Stratified** curves by protected attributes (gender, age bracket) as the first step of the fairness audit

We instantiate `KaplanMeierFitter` from `lifelines` and call `.fit()`, passing the duration array `n_survtimearr` and the event indicator array `n_eventsarr`.

The printed table shows the probability of an employee still being with the company at years 1, 2, 3, 5, 10 and 25 along with the confidence interval at each checkpoint. This directly answers the question: *"What fraction of new hires can we expect to still be here in `t` years?"*

In [ ]:
from lifelines import KaplanMeierFitter

kmf = KaplanMeierFitter(label='KM estimate')
kmf.fit(n_survtimearr, event_observed=n_eventsarr, alpha=0.05)

median_surv = kmf.median_survival_time_

print('Kaplan-Meier Estimates — Overall Population')
print()
print(f' Median survival time : {median_surv:.1f} years')

for yr in [1, 2, 3, 5, 10, 25]:
    s = kmf.predict(yr)
    ci = kmf.confidence_interval_survival_function_
    idx = ci.index.get_indexer([yr], method='pad')[0]
    l = ci.iloc[idx, 0]
    u = ci.iloc[idx, 1]
    print(f' P(stay > {yr:2d} yr) = {s:.3f}  95% CI [{l:.3f}, {u:.3f}]')

Now we produce two side-by-side panels that together give a complete picture of the Kaplan-Meier analysis.

**Left panel — Survival curve:**  
 Has the step function with a shaded 95% confidence band. Two reference lines are added manually:
- A horizontal dashed line at S(t) = 0.50 — anything below this means the majority of employees have left.
- A vertical dotted line at the median survival time.

**Right panel — At-risk table:**  
The "at-risk" count at each year is the number of employees whose recorded tenure is at least that many years (i.e. they were still present at that point in time). This shrinks both because employees leave *and* because the study window ends — this is the censoring effect. Showing these counts alongside the curve is important because the KM estimate becomes less reliable as the risk set shrinks; a steep drop based on 50 people is much less trustworthy than one based on 800.

In [ ]:
# Plot overall KM survival curve 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: Survival curve
ax = axes[0]
kmf.plot_survival_function(ax=ax, color=PALETTE[0], ci_show = True, at_risk_counts=False)
ax.axhline(0.5, color='gray', ls='--', lw=1.2, label='S(t) = 0.50')
ax.axvline(median_surv, color='red', ls=':', lw=1.5, label = f'Median = {median_surv:.1f} yr')
ax.set(xlabel='Years at Company', ylabel='Survival Probability S(t)', title='Kaplan-Meier Survival Curve', ylim=(0, 1.02))
ax.legend()

# Right: At-risk bar chart
ax2 = axes[1]
times_yr = list(range(0, 21))
at_risk_yr = [df.shape[0]] + [df.filter(pl.col('SurvivalTime') >= yr).shape[0] for yr in times_yr[1:]]

bars = ax2.bar(times_yr, at_risk_yr, color=PALETTE[0], alpha=0.7, edgecolor='white', width=0.7)

for bar, n in zip(bars, at_risk_yr):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(n), ha='center', fontsize=8)
    ax2.set(xlabel='Year', ylabel='Number at Risk', title='At-Risk Table by Year', xticks=times_yr)

plt.suptitle('Overall Kaplan-Meier Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Interpreting the shape:** A steep early drop in years 1–2 followed by a flatter tail is the typical pattern in voluntary turnover — early-career mismatches resolve quickly, and those who stay longer tend to stay much longer.

### 5.11 Stratified KM Curves: Operational Risk Factors

We now stratify by three operational factors that HR can directly act on. For each factor, the helper function `plot_km_comparison()` splits employees into groups and fits a separate KM curve per group.

The three factors chosen here are:
- **Overtime** — repeatedly identified in attrition research as a burnout driver.
- **Satisfaction level** — our composite score, split at the midpoint.
- **Job Level** — junior employees often show very different retention patterns to senior ones.

In [ ]:
def plot_km_comparison(ax, df, group_col, label_map, title, colors=None, ci_show=True):
    colors = colors or PALETTE
    groups = sorted(df[group_col].unique().to_list())

    data = {}

    for i, g in enumerate(groups):
        sub = df.filter(pl.col(group_col) == g)
        T = sub['SurvivalTime'].to_numpy()
        E = sub['EventObserved'].to_numpy()
        label = f"{label_map.get(g, g)} (n={sub.shape[0]})"

        KaplanMeierFitter(label=label).fit(T, event_observed=E).plot_survival_function(ax=ax, color=colors[i % len(colors)], ci_show=ci_show, ci_alpha=0.10)
        
        data[g] = (T, E)
        
    ax.axhline(0.5, color='gray', ls='--', lw=0.8, alpha=0.5)
    ax.set(xlabel='Years at Company', ylabel='S(t)', title=title, ylim=(0, 1.05))
    ax.legend(fontsize=8)


# Building job-level group column 
df = df.with_columns([
    pl.when(pl.col('JobLevel') == 1)
      .then(pl.lit('Job Level 1'))
      .when(pl.col('JobLevel') == 2)
      .then(pl.lit('Job Level 2'))
      .when(pl.col('JobLevel') == 3)
      .then(pl.lit('Job Level 3'))
      .when(pl.col('JobLevel') == 4)
      .then(pl.lit('Job Level 4'))
      .when(pl.col('JobLevel') == 5)
      .then(pl.lit('Job Level 5'))
      .alias('JobLevelGroup'),
])

# Building Satisfaction Level group column 
df = df.with_columns(
    pl.when(pl.col("SatisfactionMean") <= 2.5)
      .then(pl.lit(1))   
    .otherwise(pl.lit(0))  
    .alias("SatisfactionLevel")
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_km_comparison(axes[0], df, 'OverTime', {0: 'No Overtime', 1: 'Overtime'}, 'Survival by Overtime Status', [PALETTE[0], PALETTE[1]])

plot_km_comparison(axes[1], df, 'SatisfactionLevel', {0: 'Satisfied (≥2)', 1: 'Unsatisfied (<2)'}, 'Survival by Satisfaction Level', [PALETTE[0], PALETTE[1]])

plot_km_comparison(axes[2], df, 'JobLevelGroup', {v: v for v in df['JobLevelGroup'].unique().to_list()}, 'Survival by Job Level')

plt.suptitle('Stratified KM Curves — Operational Risk Factors',fontsize=14, y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

## 5.2 Log-Rank Test — Comparing Survival Curves Between Groups

The **log-rank test** is a non-parametric hypothesis test to determine whether two or more survival curves are statistically different.

### Hypotheses
- $H_0$: The survival functions of all groups are identical across all time points
- $H_1$: At least one group has a meaningfully different survival function

### Test Statistic :

$$\chi^2 = \frac{\left(\sum_i (O_{1i} - E_{1i})\right)^2}{\text{Var}}$$

Where $O_{1i}$ and $E_{1i}$ are observed and expected events in group 1 at each event time. The test is most sensitive to differences that are **proportional over time** — i.e. when the hazard ratio between groups is roughly constant.

### Fairness implication

A statistically significant log-rank result for a **protected attribute** (gender, age) does not by itself indicate discrimination — it could reflect structural factors, job type distribution, or historical hiring patterns. However, it is a red flag that demands investigation before the model is deployed for individual employee decisions.

We run the log-rank test across all major groupings: three operational factors (overtime, satisfaction, job level) and two protected-attribute groupings (gender, age bracket). For the three-level age bracket and five-level job level comparison we use `multivariate_logrank_test` from lifelines, which generalises the two-sample test.

A p-value below 0.05 is conventionally taken as evidence to reject $H_0$, meaning the two groups have statistically different survival patterns.

In [ ]:
from lifelines.statistics import logrank_test, multivariate_logrank_test

def run_logrank(df, col, g1, g2):
    m1 = df.filter(pl.col(col) == g1)
    m2 = df.filter(pl.col(col) == g2)
    lr = logrank_test(
        m1['SurvivalTime'].to_numpy(),
        m2['SurvivalTime'].to_numpy(),
        event_observed_A=m1['EventObserved'].to_numpy(),
        event_observed_B=m2['EventObserved'].to_numpy(),
    )
    return lr.test_statistic, lr.p_value


# Add Age Bracket 
df = df.with_columns([
    pl.when(pl.col('Age') < 35).then(pl.lit('Young (<35)'))
      .when(pl.col('Age') < 50).then(pl.lit('Mid (35-49)'))
      .otherwise(pl.lit('Senior (50+)'))
      .alias('AgeBracket'),
])

results = []

# Binary comparisons 
tests = [
    ('OverTime', 0, 1, 'Overtime vs No Overtime', False),
    ('SatisfactionLevel', 0, 1, 'Low vs High Satisfaction', False),
    ('Gender', 0, 1, 'Female vs Male', True),
]

for col, g1, g2, label, protected in tests:
    chi2, p = run_logrank(df, col, g1, g2)
    results.append({'Factor': label, 'chi2': chi2, 'p': p, 'Protected': protected})

# Multivariate tests 
multi_tests = [
    ('JobLevel', 'Job Level (5-way)', False),
    ('AgeBracket', 'Age Bracket (3-way)', True),
]

for col, label, protected in multi_tests:
    res = multivariate_logrank_test(
        df['SurvivalTime'].to_numpy(),
        df[col].to_numpy(),
        event_observed=df['EventObserved'].to_numpy(),
    )
    results.append({'Factor': label, 'chi2': res.test_statistic, 'p': res.p_value, 'Protected': protected})

# Results DataFrame 
lr_pl = pl.DataFrame(results).with_columns([
    pl.when(pl.col('p') < 0.001).then(pl.lit('***'))
      .when(pl.col('p') < 0.01) .then(pl.lit('**'))
      .when(pl.col('p') < 0.05) .then(pl.lit('*'))
      .otherwise(pl.lit('ns'))
      .alias('Sig'),
])

# Print 
print('Log-Rank Test :')
print()
print(f'  {"Factor":<38} {"χ²":>7}  {"p-value":>9}  {"Sig":>5}  {"Protected"}')
print()
for row in lr_pl.iter_rows(named=True):
    print(f'  {row["Factor"]:<38} {row["chi2"]:7.2f}  {row["p"]:9.4f}  {row["Sig"]:>5}  {"YES" if row["Protected"] else "no"}')

Rows marked `YES` for Protected are the ones that require fairness scrutiny. A significant(***) result there means the raw survival data already shows group differences — the model will learn from these differences and will likely reproduce them in its predictions.

## 5.3 KM Curves by Protected Attributes (Fairness Pre-Check)

Before fitting the Cox model, we visualise survival curves stratified by Gender and Age Bracket. These plots serve as the **baseline fairness reference** — they show the raw disparity in the data before any model is involved. Any model trained on this data will learn from these disparities, and the fairness audit will measure whether the model amplifies or attenuates them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Helper to fetch log-rank result
def get_lr(label):
    return lr_pl.filter(pl.col('Factor') == label).row(0, named=True)

# Add GenderLabel Column 
df = df.with_columns([
    pl.when(pl.col('Gender') == 0).then(pl.lit('Female'))
      .otherwise(pl.lit('Male'))
      .alias('GenderLabel')
])

# Gender Plot 
plot_km_comparison(
    axes[0],
    df,
    'GenderLabel',
    {'Female': "Female" , 'Male': "Male"},
    'Survival by Gender  ⚠ Protected Attribute',
    colors=[FAIR_PAL['Female'], FAIR_PAL['Male']]
)

# ── Age Bracket Plot ───────────────────
plot_km_comparison(
    axes[1],
    df,
    'AgeBracket',
    {
        'Young (<35)': 'Young (<35)',
        'Mid (35-49)': 'Mid (35-49)',
        'Senior (50+)': 'Senior (50+)'
    },
    'Survival by Age Bracket  ⚠ Protected Attribute',
    colors=[FAIR_PAL['Young'], FAIR_PAL['Mid'], FAIR_PAL['Senior']]
)

plt.suptitle('FAIRNESS PRE-CHECK — Raw Survival Disparity by Protected Attributes',fontsize=13, y=1.02, color='darkred', fontweight='bold')
plt.tight_layout()
plt.show()

**Interpreting demographic survival gaps:**

If the gender curves diverge significantly (large gap, low p-value), this tells us that female and male employees have historically left at different rates. There are many possible explanations — differences in job type, department, overtime rates, or pay equity — all of which are structural factors that the model will absorb unless we explicitly address them.

Age differences in survival are also common and reflect legitimate career-stage effects: younger employees may be more mobile, senior employees may be closer to retirement. However, a model that assigns higher predicted attrition risk to older workers could be used — intentionally or not — to justify under-investing in their development, which constitutes as age discrimination.

## 5.4 Cox Proportional Hazards Model

The Kaplan-Meier estimator and log-rank test examine one factor at a time. In practice, survival differences between groups are confounded — women may have shorter tenure not because of gender per se, but because they are more often assigned to roles with higher overtime demands. The **Cox Proportional Hazards model** allows us to estimate the independent effect of each factor while controlling for all others simultaneously.

### Model Specification

$$h(t \mid \mathbf{x}) = h_0(t) \cdot \exp\left(\beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p\right)$$

- $h_0(t)$ — the **baseline hazard**: the risk for an employee with all features at zero (or at the mean, after standardisation). It is never estimated directly — this is what makes Cox *semi-parametric*.
- $\exp(\beta_j)$ — the **hazard ratio (HR)** for feature $j$: the multiplicative change in attrition risk per one-unit (or, after standardisation, one-standard-deviation) increase in $x_j$, holding all other features constant.
- **HR > 1**: this feature increases attrition risk (faster departure)
- **HR < 1**: this feature is protective (slower departure / longer tenure)
- **HR = 1**: this feature has no effect on attrition risk

### Proportional Hazards Assumption

The model assumes that the hazard *ratio* between any two employees is **constant over time** — the curves may be at different levels, but they never cross. If this assumption is violated for a given covariate (e.g., the effect of satisfaction on attrition changes as employees age), the estimated HR for that covariate is a biased average over time.

### Fairness implication of Cox

When we include `Gender` and `Age` directly as covariates, the model learns their association with attrition *after controlling for all other features*. A large HR for Gender, even after controlling for overtime and job level, suggests a direct association that cannot be explained away by job conditions. Conversely, if the Gender HR is close to 1 after controlling for overtime, that is evidence that the gender survival gap is *mediated* by overtime allocation — a structural issue, but one that removes the basis for a gender-based prediction.

### Two model specifications

We will fit two versions of the Cox model:

1. **Full model** — all features including `Gender` and `Age`
2. **Fairness-constrained model** — protected attributes (`Gender`, `Age`) excluded as direct inputs

Comparing the hazard ratios and C-index between the two models tells us how much predictive power comes from demographic attributes vs. job-related factors. If the C-index barely changes when we drop protected attributes, we can use the fairness-constrained model with minimal accuracy cost.

In [ ]:
# Feature list — includes protected attributes Gender and Age
cox_features_full = [
    'Age', 'Gender', 'OverTime', 'MonthlyIncome', 'DistanceFromHome',
    'JobLevel', 'NumCompaniesWorked', 'YearsSinceLastPromotion',
    'TotalWorkingYears', 'TrainingTimesLastYear',
    'SatisfactionMean', 'SatisfactionMin',
    'TenureCompanyRatio', 'PromotionLagRatio', 'AvgTenurePerCompany',
    'WorkLifeBalance', 'JobInvolvement', 'PerformanceRating', 'StockOptionLevel'
]

cox_pl_full = df.select(cox_features_full + ['SurvivalTime', 'EventObserved']).fill_null(0)
X_raw_full  = cox_pl_full.select(cox_features_full).to_numpy().astype(float)

scaler_full  = StandardScaler()
X_scaled_full = scaler_full.fit_transform(X_raw_full)

cox_pd_full  = cox_pl_full.to_pandas()                   
cox_pd_full[cox_features_full] = X_scaled_full

print('Fitting Full Cox PH model (includes Gender + Age):')
cph_full = CoxPHFitter(penalizer=0.05)
cph_full.fit(cox_pd_full, duration_col='SurvivalTime', event_col='EventObserved')
print(f'  Concordance index (full model): {cph_full.concordance_index_:.4f}')

survival_curves = cph_full.predict_survival_function(cox_pd_full)

cph_full.print_summary(decimals=3, columns=['coef', 'exp(coef)', 'p', 'exp(coef) lower 95%', 'exp(coef) upper 95%'])

**What to look for in the full model summary:**  
Scan the output for the `Gender` and `Age` rows. Their `exp(coef)` values (hazard ratios) and p-values tell us the estimated effect of each protected attribute on attrition risk *after controlling for all other features*:

- If **Age** has HR < 1 with p < 0.05 after controlling for job level and income, that means younger employees (lower Age values) have higher attrition risk. This may be legitimate (career mobility) but will translate to the model predicting higher risk for younger workers.
- If **Gender** has HR noticeably different from 1 with p < 0.05 after controlling for overtime and job conditions, the model is encoding a residual gender effect that cannot be explained by observable working conditions — a fairness concern.

A concordance index value above 0.70 is generally considered good for HR data.

### 5.41 Fit the Fairness-Constrained Cox Model

We now refit the Cox model with `Gender` and `Age` removed as direct inputs. The remaining 17 features capture job characteristics, tenure patterns, and satisfaction — all factors that an HR system can legitimately act on without reference to who an employee is demographically.

This constrained model is what we would recommend for production deployment. Comparing its C-index to the full model tells us the accuracy cost of the fairness constraint.

In [ ]:
# Feature list — protected attributes excluded
cox_features_fair = [
    'OverTime', 'MonthlyIncome', 'DistanceFromHome',
    'JobLevel', 'NumCompaniesWorked', 'YearsSinceLastPromotion',
    'TotalWorkingYears', 'TrainingTimesLastYear',
    'SatisfactionMean', 'SatisfactionMin',
    'TenureCompanyRatio', 'PromotionLagRatio', 'AvgTenurePerCompany',
    'WorkLifeBalance', 'JobInvolvement', 'PerformanceRating', 'StockOptionLevel'
]

cox_pl_fair = df.select(cox_features_fair + ['SurvivalTime', 'EventObserved']).fill_null(0)
X_raw_fair = cox_pl_fair.select(cox_features_fair).to_numpy().astype(float)

scaler_fair = StandardScaler()
X_scaled_fair = scaler_fair.fit_transform(X_raw_fair)

cox_pd_fair = cox_pl_fair.to_pandas()
cox_pd_fair[cox_features_fair] = X_scaled_fair

print('Fitting Fairness-Constrained Cox PH model (Gender + Age excluded)...')
cph_fair = CoxPHFitter(penalizer=0.05)
cph_fair.fit(cox_pd_fair, duration_col='SurvivalTime', event_col='EventObserved')
print(f'  Concordance index (fair model): {cph_fair.concordance_index_:.4f}')
print()

survival_curves_fair = cph_fair.predict_survival_function(cox_pd_fair)

# Side-by-side concordance comparison
c_full = cph_full.concordance_index_
c_fair = cph_fair.concordance_index_
print('Model Comparison')
print(f'  Full model  (Gender + Age included): C-index = {c_full:.4f}')
print(f'  Fair model  (Gender + Age excluded): C-index = {c_fair:.4f}')
print(f'  Accuracy cost of fairness constraint: Δ = {c_full - c_fair:+.4f}')

**The fairness-accuracy trade-off:**  
If the C-index difference between the full and fair models is small (typically < 0.01–0.02), the protected attributes contribute very little unique predictive information beyond what the job-related features already capture. In that case, the fairness-constrained model is strongly preferred — same predictive quality, no demographic encoding.

If the gap is larger (e.g., > 0.03), the protected attributes are capturing variance that the job features don't fully explain. This could mean the feature engineering is incomplete — perhaps there are job-condition proxies for age (years of experience) or gender (department) that are already in the model and accounting for most of the effect. Investigating *why* the protected attributes contribute can lead to better job-condition features rather than falling back to demographic encoding.

### 5.42 Forest Plot of Cox Hazard Ratios (Both Models)

We visualise the hazard ratios from both models side by side. Features are sorted by the fair model's HR (ascending) so the most protective factors appear at top and the riskiest at bottom.

The `Gender` and `Age` bars appear in the full model plot (left) but are absent from the fair model plot (right). Comparing the HR values for shared features between the two models reveals whether removing the protected attributes changes the estimated effects of the remaining features (a sign of confounding).

In [ ]:
from matplotlib.patches import Patch

def make_hr_pl(cph_model, feature_list):
    hr_pd = cph_model.summary[['coef', 'exp(coef)', 'p']].reset_index()
    hr_pd.columns = ['Feature', 'Coef', 'HR', 'p']
    return pl.from_pandas(hr_pd).sort('HR')    

def draw_forest(ax, hr_df_pl, title, highlight_protected=False):
    protected = {'Age', 'Gender'}
    hr_vals = hr_df_pl.get_column('HR').to_list()          
    feat_names = hr_df_pl.get_column('Feature').to_list()
    y_pos = range(len(hr_df_pl))

    colors = []
    
    for feat, hr in zip(feat_names, hr_vals):
        if highlight_protected and feat in protected:
            colors.append('#FF6B35')   # orange = protected attribute
        elif hr > 1:
            colors.append(PALETTE[1])  # red = risk-elevating
        else:
            colors.append(PALETTE[0])  # blue = protective


    ax.barh(list(y_pos), [hr - 1 for hr in hr_vals], left=1,color=colors, alpha=0.78, edgecolor='white', height=0.72)

    ax.axvline(1, color='black', lw=1.5, ls='--', alpha=0.7)

    for i, (feat, hr) in enumerate(zip(feat_names, hr_vals)):
        ax.text(hr + 0.01, i, f'{hr:.2f}', va='center', fontsize=7.5)
        
    ax.set_yticks(list(y_pos))

    ax.set_yticklabels(feat_names, fontsize=8.5)

    ax.set(xlabel='Hazard Ratio', title=title)


hr_full_pl = make_hr_pl(cph_full, cox_features_full)
hr_fair_pl = make_hr_pl(cph_fair, cox_features_fair)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

draw_forest(axes[0], hr_full_pl, f'Full Model (C={c_full:.3f}) — incl. Gender & Age', highlight_protected=True)

draw_forest(axes[1], hr_fair_pl, f'Fair Model (C={c_fair:.3f}) — Gender & Age excluded')

# Legend
legend_els = [
    Patch(facecolor=PALETTE[1], alpha=0.78, label='HR > 1 (Higher Attrition Risk)'),
    Patch(facecolor=PALETTE[0], alpha=0.78, label='HR < 1 (Protective)'),
    Patch(facecolor='#FF6B35',  alpha=0.78, label='Protected Attribute (full model only)'),
]

axes[0].legend(handles=legend_els, fontsize=8, loc='lower right')

plt.suptitle('Cox PH Hazard Ratios — Full vs Fairness-Constrained Model', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**What the forest plots reveal:**  
Focus on two things:

1. **The orange bars (protected attributes in the left panel).** If `Gender` or `Age` have HRs substantially different from 1.0 in the full model, removing them is genuinely removing a source of demographic signal from the predictions. If they are close to 1.0 with wide CIs, they were contributing little anyway and their removal has minimal fairness benefit.

2. **Changes in the other HRs between panels.** If `OverTime` or `SatisfactionMean` shift noticeably when Gender/Age are removed, those features were acting partly as demographic proxies in the full model. In the fair model, their HRs reflect their true independent contribution to attrition, free from the confounding effect of demographic correlation.

## 5.5 Individual Risk Scores and Flight-Risk Tiers

We use the **fairness-constrained model** (`cph_fair`) to generate individual predicted 1-year attrition probabilities for every employee.

`cph_fair.predict_survival_function(X, times=[1.0])` evaluates the Cox-Breslow survival estimate at exactly 1 year for every employee. The 1-year probability is then used to assign each employee to one of three risk tiers.

In [ ]:
# Predict 1-year survival using the fair model
cox_X_fair  = cox_pd_fair.drop(columns=['SurvivalTime', 'EventObserved'])
sf_1yr      = cph_fair.predict_survival_function(cox_X_fair, times=[1.0])
prob_1yr    = (1.0 - sf_1yr.T.values.ravel()).clip(0, 1)  # 1-year attrition prob

cox_hazard  = np.exp(cph_fair.predict_log_partial_hazard(cox_X_fair).values)

# Attach scores + protected attribute labels
risk_pl = (
    df.select(['Attrition', 'SurvivalTime', 'EventObserved',
               'GenderLabel', 'AgeBracket',
               'OverTime', 'JobLevel'])
    .with_columns([
        pl.Series('AttritionProb1Yr', prob_1yr),
        pl.Series('CoxHazardScore',cox_hazard),
    ])
    .with_columns([
        pl.when(pl.col('AttritionProb1Yr') > 0.35).then(pl.lit('High Risk'))
          .when(pl.col('AttritionProb1Yr') > 0.15).then(pl.lit('Moderate Risk'))
          .otherwise(pl.lit('Low Risk'))
          .alias('RiskTier')
    ])
)

# Tier summary 
tier_summary = (
    risk_pl.group_by('RiskTier')
    .agg([
        pl.len().alias('Count'),
        pl.col('Attrition').mean().alias('ActualAttritionRate'),
        pl.col('AttritionProb1Yr').mean().alias('AvgPredictedProb'),
    ])
    .sort('ActualAttritionRate', descending=True)
)

print('Flight-Risk Tier Summary (Fair Model)')
print('='*58)
print(f'  {"Tier":<16} {"Count":>6}  {"Actual Attr%":>13}  {"Avg Pred Prob"}')
print('  ' + '-'*54)
for row in tier_summary.iter_rows(named=True):
    print(f'  {row["RiskTier"]:<16} {row["Count"]:>6}  'f'{row["ActualAttritionRate"]*100:>12.1f}%  {row["AvgPredictedProb"]:.3f}')

**Model calibration check:**  
A well-calibrated model should show that actual attrition rates increase monotonically from Low → Moderate → High Risk tiers.

## 5.51 Individual Survival Curve Visualization 
 This plot shows the predicted survival curve given a single employee id using
 the fairness-constrained Cox Proportional Hazards model `cph_fair`.

 The curve represents the survival probability `S(t)` that the employee remains
 in the company beyond time `t` (in years).

In [ ]:
employee_ids = np.arange(sf_1yr.shape[1])
sf_1yr.columns = employee_ids

def plot_employee_curve(emp_id):
    plt.figure(figsize=(6, 4))
    plt.step(times, sf_full[emp_id], where='post')
    plt.title(f'Employee {emp_id}')
    plt.xlabel('Years')
    plt.ylabel('Survival Probability S(t)')
    plt.ylim(0, 1.05)
    plt.show()

---
## 5.6 Fairness Audit

With the model fitted and risk scores computed, we now conduct a structured fairness audit. The goal is to determine whether the fairness-constrained Cox model produces predictions that are equitable across protected groups, even though `Gender` and `Age` were excluded as direct inputs.

### Why fairness auditing is necessary even after removing protected attributes :

Removing `Gender` and `Age` from the model does not guarantee fairness. Many of the remaining features are **correlated** with protected attributes:

- `MonthlyIncome` correlates with age (older workers tend to earn more) and with gender (pay gap).
- `JobLevel` correlates with gender (glass ceiling effects) and age (career progression).
- `TotalWorkingYears` is a proxy for age.
- `OverTime` may be distributed unevenly across genders.

When these proxy features remain in the model, they can reconstruct demographic patterns even without the explicit protected attribute columns present. This is called **proxy discrimination**.

### Fairness metrics we will compute :

| Metric | Definition | What a disparity means |
|---|---|---|
| **Demographic parity** | Mean predicted risk score per group | Model systematically scores one group higher |
| **High-risk rate parity** | % of each group flagged as High Risk | One group is disproportionately flagged |
| **Calibration parity** | Actual attrition rate vs predicted prob within each tier, per group | Model is well-calibrated for one group but not another |
| **C-index per group** | Discrimination ability within each group | Model ranks one group's employees more accurately than another's |

**No single metric is sufficient.** A model can pass demographic parity while failing calibration. We report all four and look for a consistent pattern.

### 5.61 Demographic Parity and High-Risk Rate by Group

We now compute summary statistics per demographic group. The key metrics are:
- Mean predicted 1-year attrition probability (should be similar across groups if the model has demographic parity).
- Proportion flagged as High Risk (the most consequential tier for HR decisions).
- Actual observed attrition rate (as a ground-truth reference point).

The **disparity ratio** — the ratio of the highest to lowest group mean score — is a commonly used single-number fairness summary. A ratio above 1.25 is a common threshold for flagging potential disparate impact.

In [ ]:
def fairness_summary(group_col, label):
    summary = (
        risk_pl
        .group_by(group_col)
        .agg([
            pl.len().alias('n'),
            pl.col('AttritionProb1Yr').mean().alias('MeanPredProb'),
            pl.col('Attrition').mean().alias('ActualAttrRate'),
            (pl.col('RiskTier') == 'High Risk').mean().alias('HighRiskRate'),
        ])
        .sort(group_col)
    )

    probs = summary['MeanPredProb']
    min_prob, max_prob = probs.min(), probs.max()
    disparity_ratio = max_prob / min_prob if min_prob else float('inf')

    print(f'\n{label} — Fairness Summary :')
    print()
    print(f'  {group_col:<18} {"n":>5}  {"Actual Attr%":>13}  ' f'{"Mean Pred Prob":>15}  {"High Risk %":>11}')

    for row in summary.iter_rows(named=True):
        print(f'  {row[group_col]:<18} {row["n"]:>5} ' f'{row["ActualAttrRate"]*100:>12.1f}%  ' f'{row["MeanPredProb"]:>14.3f}  ' f'{row["HighRiskRate"]*100:>10.1f}%')

    print(
        f'\n  Disparity ratio (max/min mean pred prob): {disparity_ratio:.3f}', ' - FLAG' if disparity_ratio > 1.25 else '  - OK')

    return summary


gender_summary = fairness_summary('GenderLabel', 'GENDER FAIRNESS AUDIT')
age_summary = fairness_summary('AgeBracket', 'AGE BRACKET FAIRNESS AUDIT')

**Interpreting demographic parity results:**

A **disparity ratio above 1.25** suggests the model assigns meaningfully higher risk scores to one group than another. This warrants investigation but is not automatically evidence of discrimination — it could be that one demographic group genuinely has higher attrition rates in this organisation, and the model is accurately reflecting that.

The critical question is: **is the disparity in predicted scores proportional to the disparity in actual attrition?** If female employees have 22% actual attrition and male employees have 18%, but the model predicts 35% for female and 20% for male, the model is *amplifying* the disparity beyond what the data supports. Conversely, if the model's predicted gap closely tracks the actual gap, it may be well-calibrated even while showing demographic disparity.

If one group has twice the High Risk flagging rate of another, HR decisions based on this tier (reduced training, performance review initiation) will be applied disproportionately to that group — a potential disparate impact concern regardless of calibration.

### 5.62 Calibration by Subgroup

Demographic parity tests whether the score distributions are similar. **Calibration** tests whether those scores are *accurate* for each group — i.e., when the model predicts 30% 1-year attrition for a group of female employees, do roughly 30% of them actually leave within a year?

We compute this within each risk tier and group combination. A model that is well-calibrated overall but poorly calibrated for a subgroup is particularly dangerous — it gives the *appearance* of accuracy while producing unreliable predictions for the disadvantaged group.

We visualise calibration as a grouped bar chart. Bars of similar height indicate good calibration; gaps indicate miscalibration.

In [ ]:
tier_order = ['High Risk', 'Moderate Risk', 'Low Risk']

def calibration_by_group(group_col, group_vals, group_colors, suptitle):
    fig, axes = plt.subplots(1, len(group_vals), figsize=(5 * len(group_vals), 5), sharey=True)
    if len(group_vals) == 1:
        axes = [axes]

    for ax, (gval, gcol) in zip(axes, zip(group_vals, group_colors)):
        sub = risk_pl.filter(pl.col(group_col) == gval) 

        actual_rates = []
        pred_rates   = []
        
        for tier in tier_order:
            tier_sub = sub.filter(pl.col('RiskTier') == tier)
            n = tier_sub.shape[0]
            if n == 0:
                actual_rates.append(0.0)
                pred_rates.append(0.0)
            else:
                actual_rates.append(tier_sub['Attrition'].mean())
                pred_rates.append(tier_sub['AttritionProb1Yr'].mean())

        x = np.arange(3)
        
        ax.bar(x - 0.18, [r * 100 for r in actual_rates], 0.35, label='Actual', color=gcol, alpha=0.85, edgecolor='white')
        ax.bar(x + 0.18, [r * 100 for r in pred_rates],   0.35, label='Predicted', color='#607D8B', alpha=0.75, edgecolor='white', hatch='//')

        ax.set_xticks(x)
        ax.set_xticklabels(['High', 'Mod.', 'Low'], fontsize=9)
        ax.set_title(f'{gval}  (n={sub.shape[0]})', fontsize=11)
        ax.set_ylabel('Attrition Rate (%)') if ax == axes[0] else None
        ax.set_ylim(0, 80)
        ax.legend(fontsize=8)

        # Max calibration gap annotation
        max_gap = max(abs(a - p) * 100 for a, p in zip(actual_rates, pred_rates))
        flag = ' *' if max_gap > 10 else ' OK'
        ax.text(0.5, 0.96, f'Max gap: {max_gap:.1f}pp{flag}', transform=ax.transAxes, ha='center', va='top', 
                fontsize=8.5, color='darkred' if max_gap > 10 else 'darkgreen',
                bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.85)
               )

    
    plt.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


calibration_by_group(
    'GenderLabel', ["Female" , "Male"],
    [FAIR_PAL['Female'], FAIR_PAL['Male']],
    'CALIBRATION AUDIT — Actual vs Predicted Attrition by Gender'
)

calibration_by_group(
    'AgeBracket', ['Young (<35)', 'Mid (35-49)', 'Senior (50+)'],
    [FAIR_PAL['Young'], FAIR_PAL['Mid'], FAIR_PAL['Senior']],
    'CALIBRATION AUDIT — Actual vs Predicted Attrition by Age Bracket'
)

**Interpreting the calibration plots:**

For each demographic group, compare the solid bar (actual attrition) to the hatched bar (predicted probability) within each tier. The annotation shows the maximum calibration gap across the three tiers:

- **Gap > 10 percentage points:** The model is meaningfully miscalibrated for this group in at least one tier. If this happens for `Senior (50+)` employees in the High Risk tier — meaning the model predicts 45% but only 28% actually leave — then Senior employees are being over-flagged as flight risks. HR decisions based on these flags (reduced training budget, performance scrutiny) would harm Senior employees disproportionately.

- **Gap ≤ 10pp:** The model is reasonably calibrated for this group. Predictions can be used with normal confidence.

### 5.63 C-Index Per Demographic Group

The final component of the fairness audit is **discrimination performance by group**. A model that ranks employees accurately overall but does so well for one group and poorly for another is providing unequal quality of predictions.

We compute C-index separately for each gender and age bracket using `lifelines.utils.concordance_index(event_times, predicted_scores, event_observed)`.

In [ ]:
def c_index_by_group(group_col, group_vals, label):
    print(f'\nC-Index by {label}:')
    print()
    print(f'  {group_col:<20} {"n":>5}  {"C-index":>9}  {"Rating"}')

    c_records = []

    for gval in group_vals:
        sub = risk_pl.filter(pl.col(group_col) == gval)

        if sub.height < 20:
            print(f'  {gval:<20}  — (too few events to compute)')
            continue

        T = sub['SurvivalTime'].to_numpy()
        E = sub['EventObserved'].to_numpy()
        S = -sub['CoxHazardScore'].to_numpy()

        c = concordance_index(T, S, E)

        rating = ('Excellent' if c >= 0.80 else 'Good' if c >= 0.70 else 'Moderate'  if c >= 0.60 else 'Poor')

        print(f'  {gval:<20} {sub.height:>5}  {c:>9.4f}  {rating}')
        c_records.append({'Group': str(gval), 'C_index': c, 'n': sub.height})

    # Overall reference
    c_all = concordance_index(risk_pl['SurvivalTime'].to_numpy(), -risk_pl['CoxHazardScore'].to_numpy(), risk_pl['EventObserved'].to_numpy())

    print(f'{"Overall (all employees)":<20} {risk_pl.height:>5} {c_all:>9.4f}  (reference)')

    if len(c_records) >= 2:
        c_vals = [r['C_index'] for r in c_records]
        c_gap = max(c_vals) - min(c_vals)

        print(f'\n  C-index gap (max − min across groups): {c_gap:.4f}', ' - FLAG' if c_gap > 0.05 else ' - OK')

c_index_by_group('GenderLabel', ["Female", "Male"],  'Gender')
c_index_by_group('AgeBracket',  ['Young (<35)', 'Mid (35-49)', 'Senior (50+)'], 'Age Bracket')

**Interpreting C-index gaps:**

A C-index gap greater than 0.05 between demographic groups is a meaningful signal that the model performs unequally across groups. In practice this means:

- For the group with the **lower C-index**, the model's ranking of who is most likely to leave is less reliable. HR actions taken on the basis of these scores (e.g., targeting the top 10% highest-risk employees for retention conversations) will be less accurately targeted for this group.
- This can arise because the training data for that group is smaller (less statistical power), or because the features used in the model capture job conditions that are distributed differently in that group, or because attrition dynamics genuinely differ across groups in ways the current features don't fully capture.

## 5.7 Fairness Audit Summary

A four-panel visual summary of the complete fairness audit, bringing together all the metrics into a single reference chart.

The four panels are:
1. **Mean predicted risk by gender** — demographic parity check
2. **High-risk flagging rate by gender** — disparate impact check
3. **Mean predicted risk by age bracket** — age parity check
4. **Actual vs predicted in High Risk tier by gender** — calibration check for the most consequential tier

In [ ]:
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.4)

# Panel 1: Mean predicted vs actual (Gender)
ax1 = fig.add_subplot(gs[0, 0:2])

for row in gender_summary.iter_rows(named=True):
    g = row['GenderLabel']
    c = FAIR_PAL[g]

    ax1.bar(g, row['MeanPredProb'] * 100, color=c, alpha=0.85, edgecolor='white', width=0.5)
    ax1.bar(g, row['ActualAttrRate'] * 100, color=c, alpha=0.35, edgecolor=c, width=0.5, fill=False, linewidth=2)

ax1.set(ylabel='1-Year Attrition Rate (%)',
        title='Mean Predicted vs Actual Prob — Gender')

ax1.legend(handles=[
    Patch(facecolor='gray', alpha=0.85, label='Predicted (filled)'),
    Patch(facecolor='gray', alpha=0.35, label='Actual (outline)')
], fontsize=8)

# Panel 2: High-risk rate (Gender) 
ax2 = fig.add_subplot(gs[0, 2:4])

for row in gender_summary.iter_rows(named=True):
    g = row['GenderLabel']
    ax2.bar(g, row['HighRiskRate'] * 100, color=FAIR_PAL[g], alpha=0.85, edgecolor='white', width=0.5)

ax2.axhline(gender_summary['HighRiskRate'].mean() * 100, color='gray', ls='--', lw=1.2)

ax2.set(ylabel='% Flagged as High Risk', title='High-Risk Flagging Rate by Gender  (Disparate Impact Check)')
ax2.legend(fontsize=8)

#  Panel 3: Mean predicted vs actual (Age) 
ax3 = fig.add_subplot(gs[1, 0:2])

for row, c in zip(age_summary.sort('AgeBracket').iter_rows(named=True), [FAIR_PAL['Young'], FAIR_PAL['Mid'], FAIR_PAL['Senior']]):

    ax3.bar(row['AgeBracket'], row['MeanPredProb'] * 100, color=c, alpha=0.85, edgecolor='white', width=0.5)
    ax3.bar(row['AgeBracket'], row['ActualAttrRate'] * 100, color=c, alpha=0.35, edgecolor=c, width=0.5, fill=False, linewidth=2)

ax3.set(ylabel='1-Year Attrition Rate (%)', title='Mean Predicted vs Actual Prob — Age Bracket')
ax3.tick_params(axis='x', labelsize=8)

# Panel 4: High Risk calibration (Gender)
ax4 = fig.add_subplot(gs[1, 2:4])

x = np.arange(2)
genders = ["Female", "Male"]

actual_hr = [
    risk_pl.filter((pl.col('GenderLabel') == g) & (pl.col('RiskTier') == 'High Risk'))['Attrition'].mean() * 100
    if risk_pl.filter((pl.col('GenderLabel') == g) & (pl.col('RiskTier') == 'High Risk')).height else 0
    for g in genders
]

pred_hr = [
    risk_pl.filter((pl.col('GenderLabel') == g) &
                   (pl.col('RiskTier') == 'High Risk'))['AttritionProb1Yr'].mean() * 100
    if risk_pl.filter((pl.col('GenderLabel') == g) &
                      (pl.col('RiskTier') == 'High Risk')).height else 0
    for g in genders
]

ax4.bar(x - 0.18, actual_hr, 0.35, color=[FAIR_PAL['Female'], FAIR_PAL['Male']], alpha=0.85, edgecolor='white', label='Actual')

ax4.bar(x + 0.18, pred_hr, 0.35, color='#607D8B', alpha=0.75, edgecolor='white', hatch='//', label='Predicted')

ax4.set_xticks(x)
ax4.set_xticklabels(genders)
ax4.set(ylabel='Attrition Rate (%)', title='High Risk Tier Calibration by Gender')
ax4.legend(fontsize=8)

# Title 
fig.suptitle( 'FAIRNESS AUDIT SUMMARY — Fair Model (Gender & Age Excluded)', fontsize=14, fontweight='bold', color='darkred', y=1.01)

plt.show()

Interpreting the four panels together:

- If **Panels 1 and 2** show similar bar heights across gender → demographic parity is approximately achieved by the fair model
- If **Panel 2** shows one gender flagged at much higher rates than population average → potential disparate impact, even without explicit gender encoding
- If **Panel 3** shows Senior employees with notably different predicted vs actual bars → age-related miscalibration
- If **Panel 4** shows a large gap between actual (solid) and predicted (hatched) bars for one gender → that gender's High Risk employees are either being over-flagged (predicted > actual) or under-flagged (actual > predicted)

The presence of proxy discrimination — where the model reproduces demographic gaps through correlated features — is evidenced by continued disparity in Panels 2 and 3 even after removing Gender and Age. If substantial disparity remains, the recommended path is:

1. **Feature review:** Identify which remaining features carry the strongest demographic correlations (`TotalWorkingYears` for age, `MonthlyIncome` for both).
2. **Fairness constraint:** Apply in-processing constraints (adversarial debiasing or reweighting) rather than post-hoc score adjustment.
3. **Separate thresholds:** Use group-specific risk tier thresholds calibrated to achieve equal false-positive rates across groups.

## 5.8 Conclusion

### Summary of Methods:

 Method | Key Result |
|---|---|
| Kaplan-Meier | Population-level survival curves; highest attrition in years 1–2 |
| Stratified KM | Overtime and low satisfaction associated with sharply lower survival |
| Log-Rank Test | Operational factors: significant differences; protected attributes: inspected |
| Cox Full Model | Identifies hazard ratios with Gender and Age included |
| Cox Fair Model | Drops protected attributes; small C-index cost if job features are rich |

### Top Findings:

1. **Overtime** is the strongest modifiable operational risk factor: employees working overtime show significantly shorter survival times and elevated Cox hazard ratios.
2. **Satisfaction** is the strongest protective operational factor: the mean score distinguishes high-retention from high-turnover employees.
3. **First two years** are the highest-risk window: both the KM curve and the at-risk table show the steepest drop in retention in years 1–2, indicating that onboarding and early-career investment has the highest marginal retention ROI.
4. **Fairness-constrained model** loses minimal predictive accuracy while eliminating direct demographic encoding — the preferred deployment option.
5. **Proxy discrimination may persist** through correlated features.

